# **TAS (Tele Assistance System) Dimensional Analysis Modelling**
NOTE: _[DASA CASE STUDY 1]_

## **Summary**

This notebook is focused on three main objectives:
1. summarizing the key aspects of the Tele Assistance System (TAS) architecture and its adaptive capabilities in the context of telehealth services for chronic patients.
2. Modelling the TAS architecture using appropriate design notations and tools to visualize its components and interactions.
3. Simulating the TAS behavior under different scenarios to evaluate its performance and adaptability with Queue Network (QN) models.

The results will be used to evaluate the Dimensional Analysis for Software Architecture (DASA) methodology, its software tool (PyDASA) and its effectiveness in modelling and Quality Scenarios (QS) trade-off in self-adaptive-systems (SAS).

---

## **Software Architecture**
- TAS (Tele Assistance System) operates in a dynamic environment where service quality, availability, and user needs frequently change.
- The TAS is further subdivided into Controller and Target System subsystem components.
- The Controller is responsible for managing the overall system behavior, while the Target System focuses on executing specific tasks related to patient care.
- The TAS target systems follows a Service-oriented architecture (SOA) pattern.
- The TAS Controller follows a MAPE-K (Monitor-Analyze-Plan-Execute-Knowledge) feedback loop for self-adaptation.
- Adaptations focus on maintaining **reliability**, **performance**, and **compliance** with patient care standards (5 specific scenarios).
- ActivFORMS provides the runtime framework for model-based adaptation using runtime models, simulations, and verified decision-making.

---

_**NOTE: MORE DETAILS ON THE ARCHITECTURE IN THE ANALYTICAL MODELLING NOTEBOOK!.**_

---

## **Code**

_**SUMMARY:**_

This code is for the analytical solution of the Case Study (TAS) Dimensional Analysis Model and is structured as follows:
1. Analytical Dimensional Model (DA).
2. Importing necessary libraries and modules.
3. Loading DA default configuration.
4. Solving the DA analytically.
5. Solving the DA with Monte Carlo simulations.
6. Plotting the DA with the obtained metrics.
7. Loading DA 'optimal' configuration.
8. Solving the DA optimally.
9. Simulating the DA with the 'optimal' configuration.
10. Plotting the optimal DA with the obtained metrics.
11. Saving the results.
12. Comparing the analytical results (Default Vs. Optimal)
13. Visualizing the results.
14. Generating a summary report.

## **Target System Queue Network Model**

<svg viewBox="0 0 4650 2000" width="1400" height="650">
    <!-- SVG content -->
    <image href="assets/cs1/img/04A - Queue Network.svg" alt="queue-net-diagram" />
    <div align="center"><em>Image 4. TAS Queue Network Diagram.</em></div>
</svg>

### **Necessary Imports**

In [ ]:
# -*- coding: utf-8 -*-
# Native imports
import os
import re
import sys
import time
from typing import Union

# Third-party imports
import numpy as np
import pandas as pd

# import qimensional model libraries PyDASA
# config module
from pydasa.utils import config
# FDU modules
from pydasa.core.fundamental import Dimension
# FDU regex management
from pydasa.dimensional.framework import DimScheme
# Variable and Variable modules
from pydasa.core.parameter import Variable
# Dimensional Matrix Modelling module
from pydasa.dimensional.model import DimMatrix
# sensitivity analysis modules
from pydasa.analysis.scenario import DimSensitivity
from pydasa.handlers.influence import SensitivityHandler
# Monte Carlo Simulation modules
from pydasa.analysis.simulation import MonteCarloSim
from pydasa.handlers.practical import MonteCarloHandler

# import queue network + models packages
from src.model.queueing import Queue
from src.model.analytical import solve_jackson_network, calculate_net_metrics

# import plot functions + grahics
from src.view.plots import plot_queue_network
from src.view.plots import plot_net_comparison
from src.view.plots import plot_net_difference
from src.view.plots import plot_nodes_heatmap
from src.view.plots import plot_nodes_diffmap


### **Function Definitions**

In [ ]:
# Simple formatter for console output

def fmt(val: Union[int, float, np.number]) -> Union[str, np.ndarray]:
    """Format a number to 4 decimal places for console output.

    Args:
        val (Union[int, float, np.number, np.ndarray]): The value to format.

    Returns:
        Union[str, np.ndarray]: The formatted value as a string or an array of strings.
    """
    if isinstance(val, (int, float, np.number)):
        if np.isnan(val) or np.isinf(val):
            return str(val)
        return f"{val:.4f}"
    elif isinstance(val, np.ndarray):
        return np.array([fmt(x) for x in val])
    return val

In [ ]:
# Load configuration from a CSV file
def load(path: str, fname: str) -> pd.DataFrame:
    """Load configuration from a CSV file.

    Args:
        path (str): The directory path where the CSV file is located.
        fname (str): The name of the CSV file to load.

    Returns:
        pd.DataFrame: A DataFrame containing the configuration data.
            CSV format:
                - node: <node_id>
                - miu: <mean_service_time>
                - c: <service_channels>
                - K: <buffer_capacity | max_queue_length>
                - lambda0: <initial_arrival_rate>
                - L0: <initial_queue_length>
                - pm: <matrix_routing_probabilities>
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Loading configuration from: {_file_path}")
    df = pd.read_csv(_file_path)
    return df

In [ ]:
# save dataframes in CSV files
def save(path: str, fname: str, data: pd.DataFrame) -> None:
    """Save a DataFrame to a CSV file.

    Args:
        path (str): The directory path where the CSV file will be saved.
        fname (str): The name of the CSV file to save.
        data (pd.DataFrame): The DataFrame containing the data to save.
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Saving data to: {_file_path}")
    data.to_csv(_file_path, index=False)

In [ ]:
# path = os.path.dirname(__file__)\
PATH = os.getcwd()
print(f"Notebook path: {PATH}")

Notebook path: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies


In [ ]:
# Folder names
asset_folder = "assets"
docs_folder = "docs"
img_folder = "img"
data_folder = "data"
report_folder = "reports"
results_folder = "results"
analysis_folder = "analysis"
cs_folder = "cs1"

In [ ]:
# setting case study data folder
file_path = os.path.join(PATH, data_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")

Data path: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies\data\cs1


In [ ]:
# load monte carlo simulation experimental data
dflt_fname = "dflt_dimensional_net_coeffs.csv"
dflt_sys_coef = load(file_path, dflt_fname)

### **Queue Model**
#### **Creating Dimensionless Charts**

##### **FDUs (Fundamental Design Units)**

##### **Base Configuration**

In [18]:
# Load configuration with mixed queue models
dflt_qn_cfg = load(file_path, "default_qn_model.csv")
print("Queue Network Configuration:")
dflt_qn_cfg.head()

Loading configuration from: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies\data\cs1\default_qn_model.csv
Queue Network Configuration:


,node,name,type,miu,s,K,lambda0,L0,pm
0,1,TAS 1(1)*,M/M/1/K,900.0,1,1000,345.0,0,"[0.00,0.75,0.25,0.00,0.00,0.00,0.00,0.00,0.00,..."
1,2,TAS 2(1)*,M/M/1/K,700.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.33,0.33,0.33,0.00,0.00,0.00,..."
2,3,TAS 3(1)*,M/M/1/K,700.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.00,0.00,0.00,0.33,0.33,0.33,..."
3,4,MAS 1,M/M/1/K,180.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.12,0.00,0.00,0.00,0.00,0.00,..."
4,5,MAS 2,M/M/1/K,530.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.00,0.07,0.00,0.00,0.00,0.00,..."


In [19]:
# Load configuration with mixed queue models
dflt_da_cfg = load(file_path, "default_dim_variables.csv")
print("Dimension Variables Configuration:")
dflt_da_cfg.head()

Loading configuration from: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies\data\cs1\default_dim_variables.csv
Dimension Variables Configuration:


,_idx,dm,_sym,_alias,_fwk,name,description,relevant,_cat,_dims,...,_min,_max,_mean,_std_units,_std_min,_std_max,_std_mean,_dist_type,_depends,_dist_params
0,1,1,\lambda_{1},lambda_tas_1,CUSTOM,TAS 1 arrival rate,Arrival rate for TAS 1,True,IN,I*T^-1,...,0.1,345,172.55,req/s,0.1,345.0,172.55,uniform,[],"{'low': 0.1, 'high': 345}"
1,2,1,\mu_{1},mu_tas_1,CUSTOM,TAS 1 service rate,Service rate for TAS 1,True,CTRL,I*T^-1,...,0.1,900,450.05,req/s,0.1,900.0,450.05,uniform,[],"{'low': 0.1, 'high': 900}"
2,3,1,err_{1},err_tas_1,CUSTOM,TAS 1 error probability,Service error probability for TAS 1,False,CTRL,I*T^-1,...,0.00%,0.00%,0,n.a.,0.0,0.0,0.00,constant,[],{'constant': 0}
3,4,1,\chi_{1},chi_tas_1,CUSTOM,TAS 1 departure rate,Departure rate for TAS 1,True,CTRL,I*T^-1,...,0.1,345,172.55,req/s,0.1,345.0,172.55,adjusted_rate,"['\mu_{1}' , 'err_{1}']",{}
4,5,1,c_{1},c_tas_1,CUSTOM,TAS 1 resourses,Number of resources allocated in TAS 1,True,IN,I,...,1,2,1.5,req,1.0,2.0,1.50,uniform_int,[],"{'low': 1, 'high': 2}"


###### **Loading Dimensional Variables**

In [20]:
print("---- Dimensional Variables by Dimensional Matrix ----")
# create a dimensional set of variables from config file
# unique dimensional matrix
dflt_dim_relevance_lt = dflt_da_cfg["dm"].unique().tolist()
# dictionary to hold dimensional variables
dflt_dim_var_groups = {}

# group by dimensional matrix
for dm in dflt_dim_relevance_lt:
    # filter by dimensional matrix number
    _vars = dflt_da_cfg[dflt_da_cfg["dm"] == dm]
    # filter by relevant attributes
    _vars = _vars[_vars["dm"] == dm]
    print(f"Dimensional Matrix: {dm}, with {_vars.shape[0]} relevant vars.")
    tdict = {}
    for var in _vars.to_dict(orient="records"):
        key = var["_sym"]
        # remove unnecessary keys
        var.pop("dm", None)
        # cast internal dict column
        if var.get("_dist_params") != None:
            data = eval(var.get("_dist_params"))
            dt = {"_dist_params": data}
            # print(data, type(data))
            var.update(dt)
        if var.get("_depends") != None:
            data = eval(var.get("_depends"))
            dt = {"_depends": data}
            # print(data, type(data))
            var.update(dt)
        # create variable instance
        tdict[key] = Variable(**var)
        # print(f"\tCreated Variable Instance: {key}: {tdict[key]}")
    # add to dictionary
    dflt_dim_var_groups[dm] = tdict

print(f"No. of Dimensional Variables Groups: {len(dflt_dim_var_groups)}")

---- Dimensional Variables by Dimensional Matrix ----
Dimensional Matrix: 1, with 11 relevant vars.
Dimensional Matrix: 2, with 11 relevant vars.
Dimensional Matrix: 3, with 11 relevant vars.
Dimensional Matrix: 4, with 11 relevant vars.
Dimensional Matrix: 5, with 11 relevant vars.
Dimensional Matrix: 6, with 11 relevant vars.
Dimensional Matrix: 7, with 11 relevant vars.
Dimensional Matrix: 8, with 11 relevant vars.
Dimensional Matrix: 9, with 11 relevant vars.
Dimensional Matrix: 10, with 11 relevant vars.
Dimensional Matrix: 11, with 11 relevant vars.
Dimensional Matrix: 12, with 11 relevant vars.
Dimensional Matrix: 13, with 11 relevant vars.
No. of Dimensional Variables Groups: 13


In [21]:
print("---- Configure Simulation Distribution Function for Variables ----")
# im tired so lambda functions are in order
for dm in dflt_dim_var_groups:
    for key, var in dflt_dim_var_groups[dm].items():
        # print(var)
        # if the dist is uniform for discrete vals like c's and K's
        if var._dist_type == "uniform_int":
            _min = var._dist_params.get("low")  # Default if missing
            _max = var._dist_params.get("high")  # Default if missing
            # Use default parameter capture to correctly bind values
            var._dist_func = lambda l=_min, h=_max: np.random.randint(l, h)
            # print(f"var sym: {var._sym}, min: {_min}, max: {_max}")

        # if the dist is shifted uniform for discrete vals like c's + K's
        if var._dist_type == "shifted_uniform_int":
            _min = var._dist_params.get("low")
            _max = var._dist_params.get("high")
            var._dist_func = lambda b, l=_min, h=_max: b + np.random.randint(l,
                                                                             h)
            
        # if the dist is uniform for continuous vals like miu's and lambda's
        if var._dist_type == "uniform":
            _low = var._dist_params.get("low")
            _high = var._dist_params.get("high")
            var._dist_func = lambda l=_low, h=_high: np.random.uniform(l, h)
            # print(f"var sym: {var._sym}, low: {_low}, high: {_high}")

        # with exponential distribution
        if var._dist_type == "exponential":
            _scale = var._dist_params.get("scale")
            var._dist_func = lambda s=_scale:  np.random.exponential(s)

        # if the dist is constant, like the error in the queue model
        if var._dist_type == "constant":
            _constant = var._dist_params.get("constant")
            var._dist_func = lambda cst=_constant: cst

        # if the dist is adjusted uniform for continous vals like chi's = lambda's * (1 - x%)
        if var._dist_type == "adjusted_rate":
            var._dist_func = lambda rate, err: adjust_rate(rate, err)
            # var._dist_func =  adjust_rate

        # if the dist is a queue theory formula
        if var._dist_type == "queue_length":
            # var._dist_func = queue_length
            var._dist_func = lambda la_z, mu, c, K: queue_length(la_z, mu, c, K)

        if var._dist_type == "queue_time":
            # var._dist_func = queue_time
            var._dist_func = lambda La_z, lq: queue_time(La_z, lq)

---- Configure Simulation Distribution Function for Variables ----


In [22]:
test = dflt_dim_var_groups[4]
mem = dict()
for t in test:
    temp = None
    print(f"{t}: {test[t]}")
    # if variable has no dependencies
    if len(test[t].depends) == 0:
        print(f"\tNo dependencies.")
        temp = test[t]._dist_func()
        mem[t] = temp
    # if variable has dependencies
    elif len(test[t].depends) > 0:
        print(f"\tDepends on: {test[t].depends}")
        params = []
        for d in test[t].depends:
            if d in mem:
                params.append(mem[d])
                # mem[d] = test[d]._dist_func()
                print(f"\t\tDependency {d}: {mem[d]}, Type: {type(mem[d])}")
            else:
                print(f"Error: Dependency {d} not found in memory.")
                params.append(None)
        print(f"\tParams: {params}")
        if None not in params:
            temp = test[t]._dist_func(*params)
            mem[t] = temp
            # else:
                # base = test[d]._dist_func()
                # mem[d] = base
            # print(f"\t\tDependency {d}: {base}, Type: {type(base)}")

        # for m in mem:
        #     tdata = dict()
        #     tdata[m] = mem[m]
        #     print(f"\tMemory {m}: {mem[m]}, Type: {type(mem[m])}")
        #     print(f"\tMemory {m}: {mem[m]}, Type: {type(mem[m])}")
    print(f"\tSample: {temp}, Type: {type(temp)}")
    
print(test["c_{4}"].dist_func(), test["\\mu_{4}"].dist_func())

\lambda_{4}: Variable(sym='\\lambda_{4}', alias='lambda_tas_4', fwk='CUSTOM', idx=34, name='TAS 4 arrival rate', description='Arrival rate for tas 4', cat='IN', dims='I*T^-1', units='req/s', std_dims='T^(-1)*I^(1)', sym_exp='T**(-1)* I**(1)', dim_col=[-1, 1, 0], min='0.1', max='97.03125', mean='97.03125', dev=None, std_units='req/s', std_min=0.1, std_max=97.03125, std_mean=97.03125, std_dev=None, step=None, std_range=array([], dtype=float64), dist_type='uniform', dist_params={'low': 0.1, 'high': 97.03125}, dist_func=<function <lambda> at 0x000001A751603520>, depends=[], relevant=True)
	No dependencies.
	Sample: 47.543749777780505, Type: <class 'float'>
\mu_{4}: Variable(sym='\\mu_{4}', alias='mu_tas_4', fwk='CUSTOM', idx=35, name='TAS 4 service rate', description='Service rate for tas 4', cat='CTRL', dims='I*T^-1', units='req/s', std_dims='T^(-1)*I^(1)', sym_exp='T**(-1)* I**(1)', dim_col=[-1, 1, 0], min='0.1', max='180', mean='90.05', dev=None, std_units='req/s', std_min=0.1, std_max=

###### **Creating Dimensional Model**

In [23]:
print("---- Creating Dimensional Model (Matrix) ----")
dflt_dim_model_groups = {}

# group by dimensional matrix
for dm in dflt_dim_relevance_lt:
    tas_vars = dflt_dim_var_groups[dm]
    print(f"Dimensional Matrix ID: {dm}, N Variables: {len(tas_vars)}")
    dflt_dim_model_groups[dm] = DimMatrix(_fwk="CUSTOM",
                                          _idx=dm,
                                          _framework=tas_scm)
    # setting Dimensional Model variables
    dflt_dim_model_groups[dm].variables = tas_vars
    _msg = "\tSetting parameters for the DA, "
    _msg += f"N Variables: {len(dflt_dim_model_groups[dm].variables)}. "
    # _msg += f"Variables: {dflt_dim_model_groups[dm].variables.keys()}"
    print(_msg)

    # setting Dimensional Model relevance list
    dflt_dim_model_groups[dm].relevant_lt = tas_vars
    _msg = "\tSetting the relevance list for the DA, "
    _msg += f"N Variables: {len(dflt_dim_model_groups[dm].relevant_lt)}. "
    # _msg += f"Variables: {dflt_dim_model_groups[dm].relevant_lt.keys()}"
    print(_msg)


---- Creating Dimensional Model (Matrix) ----
Dimensional Matrix ID: 1, N Variables: 11
	Setting parameters for the DA, N Variables: 11. 
	Setting the relevance list for the DA, N Variables: 8. 
Dimensional Matrix ID: 2, N Variables: 11
	Setting parameters for the DA, N Variables: 11. 
	Setting the relevance list for the DA, N Variables: 8. 
Dimensional Matrix ID: 3, N Variables: 11
	Setting parameters for the DA, N Variables: 11. 
	Setting the relevance list for the DA, N Variables: 8. 
Dimensional Matrix ID: 4, N Variables: 11
	Setting parameters for the DA, N Variables: 11. 
	Setting the relevance list for the DA, N Variables: 8. 
Dimensional Matrix ID: 5, N Variables: 11
	Setting parameters for the DA, N Variables: 11. 
	Setting the relevance list for the DA, N Variables: 8. 
Dimensional Matrix ID: 6, N Variables: 11
	Setting parameters for the DA, N Variables: 11. 
	Setting the relevance list for the DA, N Variables: 8. 
Dimensional Matrix ID: 7, N Variables: 11
	Setting parameter

In [24]:
print("--- Solving Dimensional Model (Matrix) ----")
# solving each dimensional model
for dm in dflt_dim_model_groups:
    print(f"Solving Dimensional Model ID: {dm}")
    # Here you would implement the logic to solve the dimensional model
    print(f"\tCreating Matrix for Dimensional Model ID: {dm}")
    dflt_dim_model_groups[dm].create_matrix()
    dflt_dim_model_groups[dm].solve_matrix()
    n = len(dflt_dim_model_groups[dm].coefficients)
    print(f"\tDimensional Model ID: {dm}, N Coefficients: {n}")
    print(f"\tFinished Solving Dimensional Model ID: {dm}")
    # print(len(dflt_dim_model_groups[dm].coefficients), "\n")
    for k, v in dflt_dim_model_groups[dm].coefficients.items():
        print(f"\t\t{k}: {fmt(v.pi_expr)}, [{v.var_dims}]")
    # print(dflt_dim_model_groups[dm].coefficients, "\n")
    # print(dflt_dim_model_groups, "\n")

--- Solving Dimensional Model (Matrix) ----
Solving Dimensional Model ID: 1
	Creating Matrix for Dimensional Model ID: 1
	Dimensional Model ID: 1, N Coefficients: 5
	Finished Solving Dimensional Model ID: 1
		\Pi_{0}: \frac{\lambda_{1}*W_{1}}{c_{1}}, [{'\\lambda_{1}': 1, 'c_{1}': -1, 'W_{1}': 1}]
		\Pi_{1}: \frac{\mu_{1}}{\lambda_{1}}, [{'\\lambda_{1}': -1, '\\mu_{1}': 1}]
		\Pi_{2}: \frac{\chi_{1}}{\lambda_{1}}, [{'\\lambda_{1}': -1, '\\chi_{1}': 1}]
		\Pi_{3}: \frac{K_{1}}{c_{1}}, [{'c_{1}': -1, 'K_{1}': 1}]
		\Pi_{4}: \frac{L_{1}}{c_{1}}, [{'c_{1}': -1, 'L_{1}': 1}]
Solving Dimensional Model ID: 2
	Creating Matrix for Dimensional Model ID: 2
	Dimensional Model ID: 2, N Coefficients: 5
	Finished Solving Dimensional Model ID: 2
		\Pi_{0}: \frac{\lambda_{2}*W_{2}}{c_{2}}, [{'\\lambda_{2}': 1, 'c_{2}': -1, 'W_{2}': 1}]
		\Pi_{1}: \frac{\mu_{2}}{\lambda_{2}}, [{'\\lambda_{2}': -1, '\\mu_{2}': 1}]
		\Pi_{2}: \frac{\chi_{2}}{\lambda_{2}}, [{'\\lambda_{2}': -1, '\\chi_{2}': 1}]
		\Pi_{3}: \

###### **Calculating Pi-Coefficients**

In [25]:

print("---- Indexing Coefficients and Sensitivity Groups ----")
# Coefficient Groups
dflt_coef_groups = {}
# Sensitivity Groups
dflt_sens_groups = {}

for dm in dflt_dim_model_groups:
    print(f"Indexing Coefficient Groups from the Dimensional Model ID: {dm}")
    # for pi, coef in dflt_dim_model_groups[dm].coefficients.items():
    # creating sensitivity group
    dflt_sens_groups[dm] = SensitivityHandler(
        _idx=dm,
        _sym=f"$SA_{{TAS_{dm}}}$",
        name=f"Sensitivity Analysis for TAS DM {dm}",
        description=f"Sensitivity Analysis TAS Dimensional Model ID {dm}.",
        _variables=dflt_dim_var_groups[dm],
        _coefficients=dflt_dim_model_groups[dm].coefficients,
    )
    # indexing coefficient groups
    keys = dflt_dim_model_groups[dm].coefficients.keys()
    values = dflt_dim_model_groups[dm].coefficients.values()
    dflt_coef_groups[dm] = dict(zip(keys, values))
    n = len(dflt_coef_groups[dm])
    print(f"\tIndexed {n} Coefficients for DM ID: {dm}\n")


---- Indexing Coefficients and Sensitivity Groups ----
Indexing Coefficient Groups from the Dimensional Model ID: 1
	Indexed 5 Coefficients for DM ID: 1

Indexing Coefficient Groups from the Dimensional Model ID: 2
	Indexed 5 Coefficients for DM ID: 2

Indexing Coefficient Groups from the Dimensional Model ID: 3
	Indexed 5 Coefficients for DM ID: 3

Indexing Coefficient Groups from the Dimensional Model ID: 4
	Indexed 5 Coefficients for DM ID: 4

Indexing Coefficient Groups from the Dimensional Model ID: 5
	Indexed 5 Coefficients for DM ID: 5

Indexing Coefficient Groups from the Dimensional Model ID: 6
	Indexed 5 Coefficients for DM ID: 6

Indexing Coefficient Groups from the Dimensional Model ID: 7
	Indexed 5 Coefficients for DM ID: 7

Indexing Coefficient Groups from the Dimensional Model ID: 8
	Indexed 5 Coefficients for DM ID: 8

Indexing Coefficient Groups from the Dimensional Model ID: 9
	Indexed 5 Coefficients for DM ID: 9

Indexing Coefficient Groups from the Dimensional Model

###### **Running Sensitivity Analysis**

In [26]:
print("---- Executing Sensitivity Analysis ----")

for dm in dflt_sens_groups:
    print(f"Executing Sensitivity Analysis for: {dm}")
    print("\tExecuting Symbolic Analysis...")
    dflt_sens_groups[dm].analyze_symbolic(val_type="mean")
    print("\tExecuting Numerical Analysis...")
    dflt_sens_groups[dm].analyze_numeric(n_samples=n_sens)
    print(f"Finishing Analysis for: {dm}\n")

---- Executing Sensitivity Analysis ----
Executing Sensitivity Analysis for: 1
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 1

Executing Sensitivity Analysis for: 2
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 2

Executing Sensitivity Analysis for: 3
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 3

Executing Sensitivity Analysis for: 4
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 4

Executing Sensitivity Analysis for: 5
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 5

Executing Sensitivity Analysis for: 6
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 6

Executing Sensitivity Analysis for: 7
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 7

Executing Sensitivity Analysis for: 8
	Executing 

In [27]:
print("---- Sensitivity Analysis Post-Processing ----")
# detailed report
# coefficient global index
i = 0

# sensitivity report statistical data
sens_records = []

# global coefficient name = coefficient formula
pi_coef = {}

for dm in dflt_sens_groups:
    print(f"Sensitivity Report for Dimensional Model ID: {dm}")
    n = len(dflt_sens_groups[dm].results)
    print(f"\tSensitivity Reports Size: {n}")
    # for key, val in dflt_sens_groups[dm].results.items():
    #     print(f"\t{key}: {val}")
    print(f"\tEnding Sensitivity report No. {dm}\n")
    # TODO complete this part later


# # iterate over monte carlo groups
# for mcg in dflt_mc_groups:
#     # monte carlo group header
#     n = dflt_mc_groups[mcg]._iterations
#     _msg = f"\tMonte Carlo Simulation Group: {mcg}, with {n} samples."
#     print(_msg)
#     _vars = dflt_mc_groups[mcg].variables.keys()
#     # _vars = ", ".join(v for v in _vars)
#     _msg = f"\tN Variables: {len(_vars)}, Var := {list(_vars)}"
#     print(_msg)

#     _coefs = dflt_mc_groups[mcg].coefficients
#     print(f"\tNo. of Pi-Coefficients in the group: {len(_coefs)}")
#     # iterating the Pi names of the group coefficients
#     for j, pi in enumerate(_coefs):
#         # getting the coefficient data
#         coef = _coefs[pi]
#         # coefficient header
#         # getting the coefficient relevant dimensional variables
#         cvars = list(coef.var_dims.keys())
#         # print(f"\t\tCoefficient: {coef.name}: {pi} = {coef.pi_expr}")
#         # print(f"\t\t\tGlobal Idx: {i}, Inputs := {cvars}, Size: {len(cvars)}")

#         # rename Pi for DataFrame column with global idx
#         pi_rename = str(pi)
#         pi_rename = pi_rename.replace(str(j), str(i))
#         # print(f"\t\t\tRenaming Coef. FROM: {pi} TO: {pi_rename}")
#         pi_coef[pi_rename] = coef.pi_expr

#         # get simulation results
#         simulations = dflt_mc_groups[mcg].get_simulation(pi)
#         # print(type(simulations))
#         if simulations:
#             # exporting results
#             results = simulations.extract_results()
#             # print(f"\t\t\tResults keys: {list(results.keys())}")
#             for key in results:
#                 new_key = key.replace(pi, pi_rename)
#                 # dictionary[new_key] = dictionary.pop(old_key)
#                 results[new_key] = results.pop(key)
#             # print(f"\t\t\tResults keys: {list(results.keys())}")

#             sens_records.update(results)

#             # exporting statistics
#             stats = {
#                 "name": pi_rename,
#                 "coef": coef.pi_expr
#             }
#             # print(f"\t\t\tStatistics before update: {stats}")
#             stats.update(simulations.statistics)
#             # print(f"\t\t\tStatistics after update: {stats}")
#             stats_records.append(stats)

#         # incrementing global index
#         i += 1
#         # print(i)
#     print(f"\tEnding Monte Carlo group No.: {mcg}...\n")

# creating DataFrame from records
dflt_node_sens = pd.DataFrame(sens_records)


---- Sensitivity Analysis Post-Processing ----
Sensitivity Report for Dimensional Model ID: 1
	Sensitivity Reports Size: 5
	Ending Sensitivity report No. 1

Sensitivity Report for Dimensional Model ID: 2
	Sensitivity Reports Size: 5
	Ending Sensitivity report No. 2

Sensitivity Report for Dimensional Model ID: 3
	Sensitivity Reports Size: 5
	Ending Sensitivity report No. 3

Sensitivity Report for Dimensional Model ID: 4
	Sensitivity Reports Size: 5
	Ending Sensitivity report No. 4

Sensitivity Report for Dimensional Model ID: 5
	Sensitivity Reports Size: 5
	Ending Sensitivity report No. 5

Sensitivity Report for Dimensional Model ID: 6
	Sensitivity Reports Size: 5
	Ending Sensitivity report No. 6

Sensitivity Report for Dimensional Model ID: 7
	Sensitivity Reports Size: 5
	Ending Sensitivity report No. 7

Sensitivity Report for Dimensional Model ID: 8
	Sensitivity Reports Size: 5
	Ending Sensitivity report No. 8

Sensitivity Report for Dimensional Model ID: 9
	Sensitivity Reports Size:

###### **Running Monte Carlo Simulation**

In [28]:
print("---- Create Monte Carlo Simulations ----")
dflt_mc_groups = {}

for dm in dflt_dim_model_groups:
    print(f"Indexing Coefficient Groups from the Dimensional Model ID: {dm}")
    # for pi, coef in dflt_dim_model_groups[dm].coefficients.items():
    # creating sensitivity group
    dflt_mc_groups[dm] = MonteCarloHandler(
        _idx=dm,
        _fwk="CUSTOM",
        _sym=f"$MC_{{TAS_{dm}}}$",
        name=f"Monte Carlo Simulation for TAS DM {dm}",
        description=f"Monte Carlo Simulation TAS Dimensional Model ID {dm}.",
        _variables=dflt_dim_model_groups[dm]._variables,
        # _variables=dflt_dim_model_groups[dm]._relevant_lt,
        _coefficients=dflt_dim_model_groups[dm].coefficients,
    )
    n = len(dflt_mc_groups[dm].coefficients)
    print(f"\tIndexed {n} Coefficients Monte Carlo: {dm}\n")

---- Create Monte Carlo Simulations ----
Indexing Coefficient Groups from the Dimensional Model ID: 1
	Indexed 5 Coefficients Monte Carlo: 1

Indexing Coefficient Groups from the Dimensional Model ID: 2
	Indexed 5 Coefficients Monte Carlo: 2

Indexing Coefficient Groups from the Dimensional Model ID: 3
	Indexed 5 Coefficients Monte Carlo: 3

Indexing Coefficient Groups from the Dimensional Model ID: 4
	Indexed 5 Coefficients Monte Carlo: 4

Indexing Coefficient Groups from the Dimensional Model ID: 5
	Indexed 5 Coefficients Monte Carlo: 5

Indexing Coefficient Groups from the Dimensional Model ID: 6
	Indexed 5 Coefficients Monte Carlo: 6

Indexing Coefficient Groups from the Dimensional Model ID: 7
	Indexed 5 Coefficients Monte Carlo: 7

Indexing Coefficient Groups from the Dimensional Model ID: 8
	Indexed 5 Coefficients Monte Carlo: 8

Indexing Coefficient Groups from the Dimensional Model ID: 9
	Indexed 5 Coefficients Monte Carlo: 9

Indexing Coefficient Groups from the Dimensional M

In [ ]:
print("---- Execute Monte Carlo Simulations ----")
for dm in dflt_mc_groups:
    print(f"\tStarting Montecarlo Simulation group: {dm}")
    dflt_mc_groups[dm]._config_distributions()
    dflt_mc_groups[dm]._config_simulations()
    dflt_mc_groups[dm].simulate(n_samples=n_exp)
    print(f"\tFinishing simulation for group: {dm}\n")

---- Execute Monte Carlo Simulations ----
	Starting Montecarlo Simulation group: 1
	Finishing simulation for group: 1

	Starting Montecarlo Simulation group: 2
	Finishing simulation for group: 2

	Starting Montecarlo Simulation group: 3
	Finishing simulation for group: 3

	Starting Montecarlo Simulation group: 4
	Finishing simulation for group: 4

	Starting Montecarlo Simulation group: 5
	Finishing simulation for group: 5

	Starting Montecarlo Simulation group: 6
	Finishing simulation for group: 6

	Starting Montecarlo Simulation group: 7
	Finishing simulation for group: 7

	Starting Montecarlo Simulation group: 8
	Finishing simulation for group: 8

	Starting Montecarlo Simulation group: 9


###### **Plotting Dimensional Model Chart**

In [ ]:
print("---- Monte Carlo Simulation Post-Processing (Exp + Stats) ----")
# detailed report
# coefficient global index
i = 0

# data for unified dataframes
exp_records = {}

# simulation statistical data
stats_records = []

# global coefficient name = coefficient formula
pi_coef = {}

# iterate over monte carlo groups
for mcg in dflt_mc_groups:
    # monte carlo group header
    n = dflt_mc_groups[mcg]._iterations
    _msg = f"\tMonte Carlo Simulation Group: {mcg}, with {n} samples."
    print(_msg)
    _vars = dflt_mc_groups[mcg].variables.keys()
    # _vars = ", ".join(v for v in _vars)
    _msg = f"\tN Variables: {len(_vars)}, Var := {list(_vars)}"
    print(_msg)

    _coefs = dflt_mc_groups[mcg].coefficients
    print(f"\tNo. of Pi-Coefficients in the group: {len(_coefs)}")
    # iterating the Pi names of the group coefficients
    for j, pi in enumerate(_coefs):
        # getting the coefficient data
        coef = _coefs[pi]
        # coefficient header
        # getting the coefficient relevant dimensional variables
        cvars = list(coef.var_dims.keys())
        # print(f"\t\tCoefficient: {coef.name}: {pi} = {coef.pi_expr}")
        # print(f"\t\t\tGlobal Idx: {i}, Inputs := {cvars}, Size: {len(cvars)}")

        # rename Pi for DataFrame column with global idx
        pi_rename = str(pi)
        pi_rename = pi_rename.replace(str(j), str(i))
        # print(f"\t\t\tRenaming Coef. FROM: {pi} TO: {pi_rename}")
        pi_coef[pi_rename] = coef.pi_expr

        # get simulation results
        simulations = dflt_mc_groups[mcg].get_simulation(pi)
        # print(type(simulations))
        if simulations:
            # exporting results
            results = simulations.extract_results()
            # print(f"\t\t\tResults keys: {list(results.keys())}")
            for key in list(results.keys()):
                new_key = key.replace(pi, pi_rename)
                # dictionary[new_key] = dictionary.pop(old_key)
                results[new_key] = results.pop(key)
            # print(f"\t\t\tResults keys: {list(results.keys())}")
            
            exp_records.update(results)

            # exporting statistics
            stats = {
                "name": pi_rename,
                "coef": coef.pi_expr
            }
            # print(f"\t\t\tStatistics before update: {stats}")
            stats.update(simulations.statistics)
            # print(f"\t\t\tStatistics after update: {stats}")
            stats_records.append(stats)

        # incrementing global index
        i += 1
        # print(i)
    print(f"\tEnding Monte Carlo group No.: {mcg}...\n")

# creating DataFrame from records
dflt_mc_exp = pd.DataFrame(exp_records)
dflt_mc_stats = pd.DataFrame(stats_records)

In [ ]:
print("\n--- Monte Carlo Simulation Experiment Results ---")
# print(dflt_analyt_nd_metrics)

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_dimensional_node_exp.csv", dflt_mc_exp)

print(dflt_mc_exp.shape)
dflt_mc_exp.info()
dflt_mc_exp.head()

In [ ]:
print("\n--- Monte Carlo Simulation Statistics ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_dimensional_node_stats.csv", dflt_mc_stats)

print(dflt_mc_stats.shape)
dflt_mc_stats.info()
dflt_mc_stats.head()

In [ ]:
print("---- Pi-Coefficients DataFrame ----")
# Extracting just the coefficients from the experimental DataFrame
pi_keys = list(pi_coef.keys())
print(f"Total No. of Pi-Coefficients: {len(pi_keys)}")
dflt_pi_coefs = pd.DataFrame(dflt_mc_exp[pi_keys])
pi_cols = {}
# renaming columns with coef = formula
for k, v in pi_coef.items():
    pi_cols[k] = f"{k}={v}"
# renaming columns
dflt_pi_coefs.rename(columns=pi_cols, inplace=True)

In [ ]:
print("\n--- Pi-Coefficients from Monte Carlo Simulation ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_dimensional_node_coeffs.csv", dflt_pi_coefs)

# checking the Pi DataFrame
print(dflt_pi_coefs.shape)
dflt_pi_coefs.info()
dflt_pi_coefs.head()

In [ ]:
# make generic columns for dimensionless coefficients
sys_name = "TAS"
sys_cols = [re.sub(r"\d+", sys_name, col) for col in pi_coef.values()]
# removing redundant entries, keeping order
sys_cols = list(dict.fromkeys(sys_cols))
for i in range(len(sys_cols)):
    formula = sys_cols[i]
    sys_cols[i] = f"\\Pi_{{{i+1}}}={formula}"
print(f"System generic columns: {len(sys_cols)}, {sys_cols}")

In [ ]:
print("---- Post-Processing System Pi-Coefficients ----")
print(f"Pi-Coeffishients shape: {dflt_pi_coefs.shape}")
# dictionary for system records
sys_records = {}

# iterate over the generic columns
for i, col in enumerate(sys_cols):
    print(f"\tSetting column: {col} in idx {i}")
    # mod = i % len(sys_cols)
    # print(f"\tidx: {col}, Sys Col: {col}, Module: {mod}")
    # iterate over the pi dataframe columns
    for j in range(i, dflt_pi_coefs.shape[1], len(sys_cols)):
        c = dflt_pi_coefs.columns[j]
        data = dflt_pi_coefs[c].values
        # print(f"\t\textracting data for column: {c}, size: {len(data)}")
        if col not in sys_records:
            # print(f"\t\tCreating new column: {col}!!!...")
            sys_records[col] = data
            # break
        elif col in sys_records:
            # print(f"\t\tAppending data to existing column: {col}")
            sys_records[col] = np.concatenate((sys_records[col], data))
            # break
    print(f"\tFinished column: {col} in idx {i}\n")

In [ ]:
print("---- Checking System Records Info ----")
for k, v in sys_records.items():
    print(f"{k}: {len(v)}")
    
print("---- System DataFrame ----")
dflt_sys_coef = pd.DataFrame(sys_records)
print(dflt_sys_coef.shape)
dflt_sys_coef.info()
dflt_sys_coef.head()

In [ ]:
# creating derived Pi-Coefficients
print("---- Creating Derived Pi-Coefficients ----")
# derived coefficients columns
# defining plot colums
plot_cols = []

# >>> working variables for derived a new coefficient
# Service Effectiveness Coef: Pi_6 = Pi_2 * Pi_3^{-1} = chi / mu
name = "Eff"
print(f"Creating System Congestion Coefficient: \\Pi_{{{name}}}")

# Pi_2 = mu / lambda
temp = f"\\Pi_{{2}}=\\frac{{\\mu_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_2 = dflt_sys_coef[temp]

# Pi_3 = chi / lambda
temp = f"\\Pi_{{3}}=\\frac{{\\chi_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_3 = dflt_sys_coef[temp]

numerator = "\\chi"
denominator = "\\mu"

# Pi_8 = chi * lambda / (mu * lambda) = chi / mu
eff_coef = f"\\eta="
eff_coef += f"\\Pi_{{{name}}}=\\frac{{{numerator}}}{{{denominator}}}"
dflt_sys_coef[eff_coef] = pi_2 * (pi_3**-1)

# >>> working variables for derived a new coefficient
# Stall Coefficient: Pi_7 = P_1 = lambda * W / c
name = "Stall"
print(f"Creating System Stall Coefficient: \\Pi_{{{name}}}")

# Pi_1 = lambda * W / c
numerator = f"\\lambda_{{{sys_name}}}*W_{{{sys_name}}}"
denominator = f"c_{{{sys_name}}}"
temp = f"\\Pi_{{1}}=\\frac{{{numerator}}}{{{denominator}}}"
pi_1 = dflt_sys_coef[temp]

# Pi_7 = Lambda * W / c
stall_coef = f"S=\\Pi_{{{name}}}=\\frac{{\\lambda*W}}{{c}}"
dflt_sys_coef[stall_coef] = pi_1

# >>> working variables for derived a new coefficient
# Occupation Coefficient: Pi_8 = Pi_5 * Pi_4^{-1} = L / K
name = "Occ"
print(f"Creating System Occupancy Coefficient: \\Pi_{{{name}}}")

# Pi_4 = L / c
temp = f"\\Pi_{{4}}=\\frac{{K_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
pi_4 = dflt_sys_coef[temp]

# Pi_5 = K / c
temp = f"\\Pi_{{5}}=\\frac{{L_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
pi_5 = dflt_sys_coef[temp]

numerator = "L"
denominator = "K"

# Pi_8 = L * c / (K * c) = L / K
occ_coef = f"O=\\Pi_{{{name}}}=\\frac{{{numerator}}}{{{denominator}}}"
dflt_sys_coef[occ_coef] = pi_5 * (pi_4**-1)

# change order to change chart distribution
# x-axis, y-axis, z-axis/contourn
plot_cols.append(occ_coef)
plot_cols.append(stall_coef)
plot_cols.append(eff_coef)

# checking derived coefficients
print(f"Derived Coefficients: {len(plot_cols)}, {plot_cols}")

# sort by first colum from lower to highest value
# dflt_sys_coef.sort_values(by=plot_cols[0], inplace=True)
# dflt_sys_coef.reset_index(drop=True, inplace=True)

In [ ]:
print("\n--- System Pi-Coefficients from Monte Carlo Simulation ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_dimensional_net_coeffs.csv", dflt_pi_coefs)
dflt_sys_coef.info()
dflt_sys_coef.head()

In [ ]:
# create plot dataframe
plot_df = pd.DataFrame(dflt_sys_coef[plot_cols])

In [ ]:
print("--- Charting Occupation and Congestion Coefficients ---")

# defining the contour lines
contour_vals = [
    0.05,   # 0
    0.1,    # 1
    0.2,    # 2
    0.3,    # 3
    0.4,    # 4
    0.5,    # 5
    0.6,    # 6
    # 0.65,   # 7
    0.7,    # 8
    0.8,    # 9
    0.9,    # 10
    0.95,   # 11
    0.99,   # 12
    # 1.0,    # 13
    # 1.2,    # 14
    # 1.5,    # 15
    # 2.0,    # 16
    # 3.0,    # 17
]

metrics = plot_df.columns.tolist()
labels = [
    "X",      # "Effectivness",
    "Y",      # "Stall",
    "Z",      # "Occupation",
]
contour_name = plot_cols[-1]
print(f"Contour levels from: {contour_name}")
print(f"No. of contour line {len(contour_vals)}")
plot_df.info()

In [ ]:
# clear plot data
# clear x-axis: occ <= 1
LIM_FLOW = 1.0
LIM_BUFFER = 1.0
print(f"cleaning invalid values in: {eff_coef}")
plot_df = plot_df[plot_df[eff_coef] <= LIM_FLOW]
print(f"Plot data with {eff_coef} < {LIM_FLOW}: {plot_df.shape}")

# clear contour/z-axis: eta <= 1
print(f"cleaning invalid values in: {occ_coef}")
plot_df = plot_df[plot_df[occ_coef] <= LIM_BUFFER]
print(f"Plot data with {occ_coef} < {LIM_BUFFER}: {plot_df.shape}")
plot_df.info()

In [ ]:
# sorting by Occ, them by Stall, then by Eff
print(f"Sorting plot data by: {plot_cols}")
plot_df.sort_values(by=plot_cols, inplace=True)
plot_df.reset_index(drop=True, inplace=True)
plot_df.info()

In [ ]:
# plotting the queue network dimensionless chart
# selecting images folder
file_path = os.path.join(PATH, results_folder, cs_folder, img_folder)
print(f"Data path: {file_path}")

# plot dimensionless system chart
title = "TAS Performance Chart: Initial Configuration"
plot_performance_coef_chart(plot_df,
                            contour_name,
                            contour_vals,
                            metrics,
                            labels,
                            title,
                            file_path,
                            "dflt_dimensional_perf_chart.png")

In [ ]:
# plot this data in regular chart
plot_df.info()
print(plot_df.describe())

In [ ]:
a =

##### **Optimized Configuration**

In [ ]:
# Load configuration with mixed queue models
file_path = os.path.join(PATH, data_folder, cs_folder)
opti_qn_cfg = load(file_path, "optimal_qn_model.csv")
print("Queue Network Configuration:")
opti_qn_cfg.head()

In [ ]:
# Load configuration with mixed queue models
opti_da_cfg = load(file_path, "optimal_dim_variables.csv")
print("Dimension Variables Configuration:")
opti_da_cfg.head()

###### **Loading Dimensional Variables**

In [ ]:
print("---- Dimensional Variables by Dimensional Matrix ----")
# create a dimensional set of variables from config file
# unique dimensional matrix
opti_dim_relevance_lt = opti_da_cfg["dm"].unique().tolist()
# dictionary to hold dimensional variables
opti_dim_var_groups = {}

# group by dimensional matrix
for dm in opti_dim_relevance_lt:
    # filter by dimensional matrix number
    _vars = opti_da_cfg[opti_da_cfg["dm"] == dm]
    # filter by relevant attributes
    _vars = _vars[_vars["dm"] == dm]
    print(f"Dimensional Matrix: {dm}, with {_vars.shape[0]} relevant vars.")
    tdict = {}
    for var in _vars.to_dict(orient="records"):
        key = var["_sym"]
        # remove unnecessary keys
        var.pop("dm", None)
        # cast internal dict column
        if var.get("_dist_params") != None:
            data = eval(var.get("_dist_params"))
            dt = {"_dist_params": data}
            # print(data, type(data))
            var.update(dt)
        tdict[key] = Variable(**var)
    # add to dictionary
    opti_dim_var_groups[dm] = tdict

print(f"No. of Dimensional Variables Groups: {len(opti_dim_var_groups)}")

In [ ]:
print("---- Configure Simulation Distribution Function for Variables ----")
# im tired so lambda functions are in order
for dm in opti_dim_var_groups:
    for key, var in opti_dim_var_groups[dm].items():
        # print(var)
        # if the distriburion is constant
        if var._dist_type == "constant":
            _constant = var._dist_params.get("constant")
            var._dist_func = lambda: _constant
        # with uniform distribution
        if var._dist_type == "uniform":
            _low = var._dist_params.get("low")
            _high = var._dist_params.get("high")
            var._dist_func = lambda: np.random.uniform(_low, _high)
        # with exponential distribution
        if var._dist_type == "exponential":
            _scale = var._dist_params.get("scale")
            var._dist_func = lambda:  np.random.exponential(_scale)


###### **Creating Dimensional Model**

In [ ]:
print("---- Creating Dimensional Model (Matrix) ----")
opti_dim_model_groups = {}

# group by dimensional matrix
for dm in opti_dim_relevance_lt:
    tas_vars = opti_dim_var_groups[dm]
    print(f"Dimensional Matrix ID: {dm}, N Variables: {len(tas_vars)}")
    opti_dim_model_groups[dm] = DimMatrix(_fwk="SOFTWARE",
                                          _idx=dm,
                                          _framework=tas_scm)
    # setting Dimensional Model variables
    opti_dim_model_groups[dm].variables = tas_vars
    _msg = "\tSetting parameters for the DA, "
    _msg += f"N Variables: {len(opti_dim_model_groups[dm].variables)}. "
    # _msg += f"Variables: {opti_dim_model_groups[dm].variables.keys()}"
    print(_msg)

    # setting Dimensional Model relevance list
    opti_dim_model_groups[dm].relevant_lt = tas_vars
    _msg = "\tSetting the relevance list for the DA, "
    _msg += f"N Variables: {len(opti_dim_model_groups[dm].relevant_lt)}. "
    # _msg += f"Variables: {opti_dim_model_groups[dm].relevant_lt.keys()}"
    print(_msg)


In [ ]:
print("--- Solving Dimensional Model (Matrix) ----")
# solving each dimensional model
for dm in opti_dim_model_groups:
    print(f"Solving Dimensional Model ID: {dm}")
    # Here you would implement the logic to solve the dimensional model
    print(f"\tCreating Matrix for Dimensional Model ID: {dm}")
    opti_dim_model_groups[dm].create_matrix()
    opti_dim_model_groups[dm].solve_matrix()
    n = len(opti_dim_model_groups[dm].coefficients)
    print(f"\tDimensional Model ID: {dm}, N Coefficients: {n}")
    print(f"\tFinished Solving Dimensional Model ID: {dm}")
    # print(len(opti_dim_model_groups[dm].coefficients), "\n")
    # print(opti_dim_model_groups[dm].coefficients, "\n")
    # print(opti_dim_model_groups, "\n")

###### **Calculating Pi-Coefficients**

In [ ]:

print("---- Indexing Coefficients and Sensitivity Groups ----")
# Coefficient Groups
opti_coef_groups = {}
# Sensitivity Groups
opti_sens_groups = {}

for dm in opti_dim_model_groups:
    print(f"Indexing Coefficient Groups from the Dimensional Model ID: {dm}")
    # for pi, coef in opti_dim_model_groups[dm].coefficients.items():
    # creating sensitivity group
    opti_sens_groups[dm] = SensitivityHandler(
        _idx=dm,
        _sym=f"$SA_{{TAS_{dm}}}$",
        name=f"Sensitivity Analysis for TAS DM {dm}",
        description=f"Sensitivity Analysis TAS Dimensional Model ID {dm}.",
        _variables=opti_dim_var_groups[dm],
        _coefficients=opti_dim_model_groups[dm].coefficients,
    )
    # indexing coefficient groups
    keys = opti_dim_model_groups[dm].coefficients.keys()
    values = opti_dim_model_groups[dm].coefficients.values()
    opti_coef_groups[dm] = dict(zip(keys, values))
    n = len(opti_coef_groups[dm])
    print(f"\tIndexed {n} Coefficients for DM ID: {dm}\n")


###### **Running Sensitivity Analysis**

In [ ]:
print("---- Executing Sensitivity Analysis ----")

for dm in opti_sens_groups:
    print(f"Executing Sensitivity Analysis for: {dm}")
    print("\tExecuting Symbolic Analysis...")
    opti_sens_groups[dm].analyze_symbolic(val_type="mean")
    print("\tExecuting Numerical Analysis...")
    opti_sens_groups[dm].analyze_numeric(n_samples=n_sens)
    print(f"Finishing Analysis for: {dm}\n")

In [ ]:
print("---- Sensitivity Analysis Post-Processing ----")
# detailed report
# coefficient global index
i = 0

# sensitivity report statistical data
sens_records = []

# global coefficient name = coefficient formula
pi_coef = {}

for dm in opti_sens_groups:
    print(f"Sensitivity Report for Dimensional Model ID: {dm}")
    n = len(opti_sens_groups[dm].results)
    print(f"\tSensitivity Reports Size: {n}")
    # for key, val in opti_sens_groups[dm].results.items():
    #     print(f"\t{key}: {val}")
    print(f"\tEnding Sensitivity report No. {dm}\n")

# creating DataFrame from records
opti_node_sens = pd.DataFrame(sens_records)


###### **Running Monte Carlo Simulation**

In [ ]:
print("---- Create Monte Carlo Simulations ----")
opti_mc_groups = {}

for dm in opti_dim_model_groups:
    print(f"Indexing Coefficient Groups from the Dimensional Model ID: {dm}")
    # for pi, coef in opti_dim_model_groups[dm].coefficients.items():
    # creating sensitivity group
    opti_mc_groups[dm] = MonteCarloHandler(
        _idx=dm,
        _fwk="SOFTWARE",
        _sym=f"$MC_{{TAS_{dm}}}$",
        name=f"Monte Carlo Simulation for TAS DM {dm}",
        description=f"Monte Carlo Simulation TAS Dimensional Model ID {dm}.",
        _variables=opti_dim_model_groups[dm]._relevant_lt,
        _coefficients=opti_dim_model_groups[dm].coefficients,
    )
    n = len(opti_mc_groups[dm].coefficients)
    print(f"\tIndexed {n} Coefficients Monte Carlo: {dm}\n")

In [ ]:
print("---- Execute Monte Carlo Simulations ----")
for dm in opti_mc_groups:
    print(f"\tStarting Montecarlo Simulation group: {dm}")
    opti_mc_groups[dm]._create_distributions()
    opti_mc_groups[dm]._create_simulations()
    opti_mc_groups[dm].simulate(n_samples=n_exp)
    print(f"\tFinishing simulation for group: {dm}\n")

###### **Plotting Dimensional Model Chart**

In [ ]:
print("---- Monte Carlo Simulation Post-Processing (Exp + Stats) ----")
# detailed report
# coefficient global index
i = 0

# data for unified dataframes
exp_records = {}

# simulation statistical data
stats_records = []

# global coefficient name = coefficient formula
pi_coef = {}

# iterate over monte carlo groups
for mcg in opti_mc_groups:
    # monte carlo group header
    n = opti_mc_groups[mcg]._iterations
    _msg = f"\tMonte Carlo Simulation Group: {mcg}, with {n} samples."
    print(_msg)
    _vars = opti_mc_groups[mcg].variables.keys()
    # _vars = ", ".join(v for v in _vars)
    _msg = f"\tN Variables: {len(_vars)}, Var := {list(_vars)}"
    print(_msg)

    _coefs = opti_mc_groups[mcg].coefficients
    print(f"\tNo. of Pi-Coefficients in the group: {len(_coefs)}")
    # iterating the Pi names of the group coefficients
    for j, pi in enumerate(_coefs):
        # getting the coefficient data
        coef = _coefs[pi]
        # coefficient header
        # getting the coefficient relevant dimensional variables
        cvars = list(coef.var_dims.keys())
        # print(f"\t\tCoefficient: {coef.name}: {pi} = {coef.pi_expr}")
        # print(f"\t\t\tGlobal Idx: {i}, Inputs := {cvars}, Size: {len(cvars)}")

        # rename Pi for DataFrame column with global idx
        pi_rename = str(pi)
        pi_rename = pi_rename.replace(str(j), str(i))
        # print(f"\t\t\tRenaming Coef. FROM: {pi} TO: {pi_rename}")
        pi_coef[pi_rename] = coef.pi_expr

        # get simulation results
        simulations = opti_mc_groups[mcg].get_simulation(pi)
        # print(type(simulations))
        if simulations:
            # exporting results
            results = simulations.extract_results()
            # print(f"\t\t\tResults keys: {list(results.keys())}")
            for key in results:
                new_key = key.replace(pi, pi_rename)
                # dictionary[new_key] = dictionary.pop(old_key)
                results[new_key] = results.pop(key)
            # print(f"\t\t\tResults keys: {list(results.keys())}")
            
            exp_records.update(results)

            # exporting statistics
            stats = {
                "name": pi_rename,
                "coef": coef.pi_expr
            }
            # print(f"\t\t\tStatistics before update: {stats}")
            stats.update(simulations.statistics)
            # print(f"\t\t\tStatistics after update: {stats}")
            stats_records.append(stats)

        # incrementing global index
        i += 1
        # print(i)
    print(f"\tEnding Monte Carlo group No.: {mcg}...\n")

# creating DataFrame from records
opti_mc_exp = pd.DataFrame(exp_records)
opti_mc_stats = pd.DataFrame(stats_records)

In [ ]:
print("\n--- Monte Carlo Simulation Experiment Results ---")
# print(opti_analyt_nd_metrics)

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_dimensional_node_exp.csv", opti_mc_exp)

print(opti_mc_exp.shape)
opti_mc_exp.info()
opti_mc_exp.head()

In [ ]:
print("\n--- Monte Carlo Simulation Statistics ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_dimensional_node_stats.csv", opti_mc_stats)

print(opti_mc_stats.shape)
opti_mc_stats.info()
opti_mc_stats.head()

In [ ]:
print("---- Pi-Coefficients DataFrame ----")
# Extracting just the coefficients from the experimental DataFrame
pi_keys = list(pi_coef.keys())
print(f"Total No. of Pi-Coefficients: {len(pi_keys)}")
opti_pi_coefs = pd.DataFrame(opti_mc_exp[pi_keys])
pi_cols = {}
# renaming columns with coef = formula
for k, v in pi_coef.items():
    pi_cols[k] = f"{k}={v}"
# renaming columns
opti_pi_coefs.rename(columns=pi_cols, inplace=True)

In [ ]:
print("\n--- Pi-Coefficients from Monte Carlo Simulation ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_dimensional_node_coeffs.csv", opti_pi_coefs)

# checking the Pi DataFrame
print(opti_pi_coefs.shape)
opti_pi_coefs.info()
opti_pi_coefs.head()

In [ ]:
# make generic columns for dimensionless coefficients
sys_name = "TAS"
sys_cols = [re.sub(r"\d+", sys_name, col) for col in pi_coef.values()]
# removing redundant entries, keeping order
sys_cols = list(dict.fromkeys(sys_cols))
for i in range(len(sys_cols)):
    formula = sys_cols[i]
    sys_cols[i] = f"\\Pi_{{{i+1}}}={formula}"
print(f"System generic columns: {len(sys_cols)}, {sys_cols}")

In [ ]:
print("---- Post-Processing System Pi-Coefficients ----")
print(f"Pi-Coeffishients shape: {opti_pi_coefs.shape}")
# dictionary for system records
sys_records = {}

# iterate over the generic columns
for i, col in enumerate(sys_cols):
    print(f"\tSetting column: {col} in idx {i}")
    # mod = i % len(sys_cols)
    # print(f"\tidx: {col}, Sys Col: {col}, Module: {mod}")
    # iterate over the pi dataframe columns
    for j in range(i, opti_pi_coefs.shape[1], len(sys_cols)):
        c = opti_pi_coefs.columns[j]
        data = opti_pi_coefs[c].values
        # print(f"\t\textracting data for column: {c}, size: {len(data)}")
        if col not in sys_records:
            # print(f"\t\tCreating new column: {col}!!!...")
            sys_records[col] = data
            # break
        elif col in sys_records:
            # print(f"\t\tAppending data to existing column: {col}")
            sys_records[col] = np.concatenate((sys_records[col], data))
            # break
    print(f"\tFinished column: {col} in idx {i}\n")

In [ ]:
print("---- Checking System Records Info ----")
for k, v in sys_records.items():
    print(f"{k}: {len(v)}")
    
print("---- System DataFrame ----")
opti_sys_coef = pd.DataFrame(sys_records)
print(opti_sys_coef.shape)
opti_sys_coef.info()
opti_sys_coef.head()

In [ ]:
# creating derived Pi-Coefficients
print("---- Creating Derived Pi-Coefficients ----")
# derived coefficients columns
# defining plot colums
plot_cols = []

# working variables for derived coefficients
print("Creating Pi-Coefficient from: \\Pi_{{4}}, \\Pi_{{5}}")
# Pi_4 = c / K
temp = f"\\Pi_{{4}}=\\frac{{c_{{{sys_name}}}}}{{K_{{{sys_name}}}}}"
pi_4 = opti_sys_coef[temp]

# Pi_5 = L / K
temp = f"\\Pi_{{5}}=\\frac{{L_{{{sys_name}}}}}{{K_{{{sys_name}}}}}"
pi_5 = opti_sys_coef[temp]

# 2 --->> Queue Occupancy Coef: Pi_7 = Pi_5 * Pi_4^{-1}
print("Creating System Obstruction Coefficient: \\Pi_{{7}}")
numerator = f"L_{{{sys_name}}}"
denominator = f"C_{{{sys_name}}}"

# Occupation Coefficient: Pi_6 = L / C
new_coef = f"\\Pi_{{Occ}}=\\frac{{{numerator}}}{{{denominator}}}"
opti_sys_coef[new_coef] = pi_5 * (pi_4**-1)
# adding to plot coefficients for later
plot_cols.append(new_coef)

# working variables for derived coefficients
print("Creating Pi-Coefficients from: \\Pi_{{1}}, \\Pi_{{2}}, \\Pi_{{3}}")

# Pi_1 = lambda * W
temp = f"\\Pi_{{1}}=\\lambda_{{{sys_name}}}*W_{{{sys_name}}}"
pi_1 = opti_sys_coef[temp]

# Pi_2 = chi / lambda
temp = f"\\Pi_{{2}}=\\frac{{\\chi_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_2 = opti_sys_coef[temp]

# Pi_3 = mu / lambda
temp = f"\\Pi_{{3}}=\\frac{{\\mu_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_3 = opti_sys_coef[temp]

# 1 --->> System/Component Congestion Coef: Pi_7 = Pi_2 * Pi_3^{-1} * Pi_1
print("Creating System Congestion Coefficient: \\Pi_{{7}}")
numerator = f"\\chi_{{{sys_name}}}*\\lambda_{{{sys_name}}}*W_{{{sys_name}}}"
denominator = f"\\mu_{{{sys_name}}}"

# Stall Coefficient Pi_7 = chi * Lambda * W / mu
new_coef = f"\\Pi_{{Stall}}=\\frac{{{numerator}}}{{{denominator}}}"
opti_sys_coef[new_coef] = pi_2 * (pi_3**-1) * pi_1
# adding to plot coefficients for later
plot_cols.append(new_coef)

# adding utilization to the dataframe, rho = Pi_3^{-1}
print("Creating Pi-Coefficient from :\\Pi_{{3}}")
# new_coef = f"\\rho_{{{sys_name}}}=\\frac{{\\lambda_{{{sys_name}}}}}{{\\mu_{{{sys_name}}}}}"
new_coef = f"\\rho"
# new_coef = "\\Pi_{{3}}^{-1}"
# new_coef += "=\\frac{1}{\\Pi_{{3}}}"
new_coef += f"=\\frac{{\\lambda_{{{sys_name}}}}}{{\\mu_{{{sys_name}}}}}"
# simp_coef = "$\\rho$"
opti_sys_coef[new_coef] = pi_3**-1
# adding to plot coefficients for later
plot_cols.append(new_coef)

In [ ]:
print("\n--- System Pi-Coefficients from Monte Carlo Simulation ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_dimensional_net_coeffs.csv", opti_pi_coefs)
opti_sys_coef.info()
opti_sys_coef.head()

In [ ]:
print("--- Charting Occupation and Congestion Coefficients ---")

# defining the contour lines
contour_vals = [
    # 0.05,   # 0
    0.1,    # 1
    0.2,    # 2
    0.3,    # 3
    0.4,    # 4
    0.5,    # 5
    0.6,    # 6
    # 0.65,   # 7
    0.7,    # 8
    0.8,    # 9
    0.9,    # 10
    # 0.95,   # 11
    # 0.99,   # 12
    1.0,    # 13
    1.2,    # 14
    1.5,    # 15
    2.0,    # 16
    3.0,    # 17
]

# 
plot_df = pd.DataFrame(opti_sys_coef[plot_cols])
metrics = plot_df.columns.tolist()
labels = [
    "Occupation",
    "Congestion",
    "Utilization",
]
contour_name = plot_cols[-1]
print(f"Contour levels from: {contour_name}")
print(f"No. of contour line {len(contour_vals)}")
plot_df.info()

In [ ]:
# plotting the queue network dimensionless chart
# selecting images folder
file_path = os.path.join(PATH, results_folder, cs_folder, img_folder)
print(f"Data path: {file_path}")

# plot dimensionless system chart
title = "TAS Performance Coefficients: After Adaptation."
plot_performance_coef_chart(plot_df,
                            contour_name,
                            contour_vals,
                            metrics,
                            labels,
                            title,
                            file_path,
                            "opti_dimensional_perf_chart.png")

#### **Solving Dimensional Model**

##### **Base Configuration**

In [ ]:
# extract parameters from the configuration DataFrame
# and casting them to proper types
nodes = list(dflt_qn_cfg["node"].values.astype(int))
names = list(dflt_qn_cfg["name"].values)
types = list(dflt_qn_cfg["type"].values)
mius = list(dflt_qn_cfg["miu"].values)
lambda_zs = list(dflt_qn_cfg["lambda0"].values)
n_servers = list(dflt_qn_cfg["s"].values.astype(int))
kaps = list(dflt_qn_cfg["K"].values.astype(float))

# Convert K=0=nan to understandable infinite capacity -> None
for i in range(len(kaps)):
    if np.isnan(kaps[i]):
        kaps[i] = None
    else:
        kaps[i] = float(kaps[i])

# Convert string representations of arrays to actual numpy arrays
# and create routing matrix P
prob = []
for pm_str in dflt_qn_cfg["pm"].values:
    pm_values = pm_str.strip("[]").split(",")
    pm_values = [float(val) for val in pm_values]
    prob.append(pm_values)
P = np.array((prob))

# create queue objects based on the configuration
queues = []
for i in range(len(mius)):
    # print(f"Creating queue: '{names[i]}' for Node: {nodes[i]}")
    # print(f"  Type: {types[i]}, λ: {lambda_zs[i]}, μ: {mius[i]}, s: {n_servers[i]}, K: {kaps[i]}")
    # create queue object
    q = Queue(
        model=types[i],
        _lambda=lambda_zs[i],
        miu=mius[i],
        n_servers=n_servers[i],
        kapacity=kaps[i],
    )
    # add queue to the list
    queues.append(q)

In [ ]:
# Solve the network analytically
# first node metrics
dflt_analyt_nd_metrics = solve_jackson_network(mius, lambda_zs, queues, P)

# then network metrics
dflt_analyt_net_metrics = calculate_net_metrics(dflt_analyt_nd_metrics)
dflt_analyt_net_metrics["nodes"] = len(list(dflt_analyt_nd_metrics["node"]))

##### **Optimized Configuration**

#### **Data Analysis**

The steps are:

1) Extract and organize simulation data into a unique DataFrame.
2) Add metadata and basic statistics.
3) Save the dataframe to a CSV file.
4) Create a basic dimensionless plot (similar to Moody's chart)

##### **Data Post-Processing**


##### **Optimized Configuration**

#### **Numerical Solution (Simulation-Based)**

##### **Base Configuration**

##### **Optimized Configuration**

## **Results**

### **Compare Results**

### **Saving Results**

## **Analysis**

### **Graph Analysis**

In [ ]:
# Create directories if they dont exist
os.makedirs("data/CS-1-HealthTAS/img", exist_ok=True)

# Figure setup with appropriate size, resolution and white background
fig, ax = plt.subplots(figsize=(12, 9), dpi=300, facecolor="white")
ax.set_facecolor("white")

# Get first two coefficients for the plot axes
pi_list = list(tas_mch._coefficients.keys())
pi_x = pi_list[0]  # / pi_list[3]  # first/Third coefficient for x-axis
pi_x1 = pi_list[3]               # First coefficient for x-axis
pi_y = pi_list[1]               # Second coefficient for y-axis

# Calculate utilization ratio (common in queueing theory)
df_results["utilization"] = 1 / df_results["\\Pi_{2}"]

# Group data by utilization ratio to create characteristic curves
utilization_ranges = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
colors = plt.cm.viridis(np.linspace(0, 1, len(utilization_ranges)))

# Add hexbin background to show data density (like Moody chart)
hb = plt.hexbin(
    df_results[pi_x],
    df_results[pi_y],
    gridsize=40,
    cmap="Blues",
    alpha=0.10,
    mincnt=1
)

# Plot characteristic curves for different utilization ranges
for i, util in enumerate(utilization_ranges):
    # Filter data near this utilization value
    mask = (df_results["utilization"] >= util - 0.05) & \
           (df_results["utilization"] <= util + 0.05)
    subset = df_results[mask]

    if len(subset) > 10:  # Only draw if we have enough points
        # Sort by x-value for smooth curves
        subset = subset.sort_values(by=pi_x)

        # Create a polynomial fit (3rd degree works well for most curves)
        if len(subset) > 5:
            z = np.polyfit(np.log10(subset[pi_x]), np.log10(subset[pi_y]), 3)
            p = np.poly1d(z)

            # Create smooth x values for the line
            x_smooth = np.logspace(
                np.log10(subset[pi_x].min()),
                np.log10(subset[pi_x].max()),
                100
            )

            # Calculate predicted y values (convert back from log space)
            y_smooth = 10**p(np.log10(x_smooth))

            # Plot the trend line
            plt.plot(
                x_smooth,
                y_smooth,
                "-",
                linewidth=2,
                color=colors[i],
                label=f"ρ = {util:.1f}"
            )

# # Add contour lines for queue length
# queue_length_values = [1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3]
# # [10, 20, 30, 40, 50]
# # [1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3]
# contour = plt.tricontour(
#     df_results[pi_x1],
#     df_results[pi_y],
#     # df_results["\\Pi_{3}"],
#     # df_results["n_{1}_\\Pi_{0}"],
#     df_results["L_{1}_\\Pi_{3}"],
#     # df_results[pi_x] / df_results[pi_x1],
#     levels=queue_length_values,
#     colors="black",
#     linestyles="dashed",
#     linewidths=0.5,
#     alpha=0.75
# )

# # Add contour labels in black
# plt.clabel(contour, inline=True, fontsize=10, fmt="n=%4.0f", colors="black")

# Set up log scales (standard for Moody-like charts)
plt.xscale("log")
plt.yscale("log")

# Add grid with minor lines - lighter color for better visibility on white background
plt.grid(True, which="both", ls="-", color="lightgray", alpha=0.75)
plt.grid(True, which="minor", ls=":", color="lightgray", alpha=0.75)
plt.minorticks_on()

# Format tick labels for better readability
formatter = ticker.LogFormatterMathtext(base=10)
plt.gca().xaxis.set_major_formatter(formatter)
plt.gca().yaxis.set_major_formatter(formatter)

# Make sure ticks and labels are black
ax.tick_params(axis="both", colors="black")

# Add descriptive labels and title in black
plt.xlabel(
    f"${pi_x}$ = ${tas_mch._coefficients[pi_x].pi_expr}$", fontsize=14, color="black")
plt.ylabel(
    f"${pi_y}$ = ${tas_mch._coefficients[pi_y].pi_expr}$", fontsize=14, color="black")
plt.title("Queue System Performance Chart", fontsize=16, color="black")

# Add legend with clear styling
legend = plt.legend(
    title="System Utilization ($\\rho$)",
    loc="best",
    fontsize=12,
    framealpha=0.9
)
legend.get_title().set_color("black")

# Add annotations for regions with better visibility on white background
plt.text(
    df_results[pi_x].min()*1.5,
    df_results[pi_y].max()*0.8,
    "Low Utilization\nRegion",
    fontsize=12,
    ha="left",
    color="black",
    bbox=dict(facecolor="white", alpha=0.95,
              boxstyle="round", edgecolor="gray")
)

plt.text(
    df_results[pi_x].max()*0.5,
    df_results[pi_y].min()*1.5,
    "High Utilization\nRegion",
    fontsize=12,
    ha="right",
    color="black",
    bbox=dict(facecolor="white", alpha=0.95,
              boxstyle="round", edgecolor="gray")
)

# Save and display with white background
plt.tight_layout()
plt.savefig("data/CS-1-HealthTAS/img/tas_behaviour_chart.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

## **Conclusion**

Understanding Contour Behavior in Your Queue System Chart
The contour lines in your plot represent queue length values (10, 20, 30, 40, 50), and they appear higher and closer to the Y-axis for higher values due to several important queue theory principles:

Why This Pattern Occurs
Exponential Queue Growth: In queueing theory, as system utilization (ρ = λ/μ) approaches 1.0, queue length grows exponentially rather than linearly. This fundamental property creates the compressed contour pattern you're seeing.

Dimensionless Variables Relationship: Your dimensionless plot shows that:

Small changes in the X-axis variable (first Pi coefficient) cause large changes in queue length when the system is near capacity
The Y-axis variable (second Pi coefficient) has less impact on queue length at higher utilization levels
Log-Log Scale Effect: You're using logarithmic scales on both axes, which compresses the higher values and spreads out the lower values, making this pattern more pronounced.

Queue Theory Interpretation
This pattern visualizes a critical queueing system concept: the performance cliff effect. As your system approaches saturation:

Small increases in load (moving right on X-axis) cause dramatically larger queue lengths
Improving service rate (moving up on Y-axis) provides diminishing returns once the system is near saturation
This is why the higher contour lines (40, 50) are compressed toward the right side of the chart and appear to "stack up" near the Y-axis.

Engineering Significance
This pattern in your data suggests:

Your system has a critical utilization threshold beyond which performance degrades rapidly
The "High Utilization Region" you've marked represents the danger zone where small load increases cause large queue growth
The system should be operated with sufficient margin from this threshold to maintain stable performance
This is exactly the kind of relationship a Moody-style chart should reveal - helping identify safe operating regions and performance boundaries in your system.

## **Future Work**

## **References & Sources**
<!-- TODO fix the references, links and details -->
1. [Queueing Theory](https://en.wikipedia.org/wiki/Queueing_theory)
2. [Dimensional Analysis](https://en.wikipedia.org/wiki/Dimensional_analysis)
3. [Simulation in Healthcare](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6466220/)

---

# **HASTA AKI!!!**

In [ ]:
import warnings
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb  # Often the best for tabular data

print("=" * 80)
print("ADVANCED REGRESSION: USING ORIGINAL Π₁-Π₅ COEFFICIENTS")
print("Hypothesis: Stall is better predicted by original dimensional coefficients")
print("=" * 80)

warnings.filterwarnings('ignore')

# ============================================================================
# EXTRACT ORIGINAL Π COEFFICIENTS FROM dflt_sys_coef
# ============================================================================

print(f"\n{'='*80}")
print("DATA PREPARATION: EXTRACTING ORIGINAL Π COEFFICIENTS")
print(f"{'='*80}\n")

# Identify original Pi coefficient columns (before derived ones)
sys_name = "TAS"
original_pi_cols = [
    # f"\\Pi_{{1}}=\\frac{{\\lambda_{{{sys_name}}}*W_{{{sys_name}}}}}{{c_{{{sys_name}}}}}",  # Π₁
    # # Π₂
    # f"\\Pi_{{2}}=\\frac{{\\mu_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}",
    # # Π₃
    # f"\\Pi_{{3}}=\\frac{{\\chi_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}",
    # # Π₄
    # f"\\Pi_{{4}}=\\frac{{K_{{{sys_name}}}}}{{c_{{{sys_name}}}}}",
    # # Π₅
    # f"\\Pi_{{5}}=\\frac{{L_{{{sys_name}}}}}{{c_{{{sys_name}}}}}",

    # Occupancy
    f"O=\\Pi_{{Occ}}=\\ln(\\frac{{L}}{{K}})",
    # Efficiency
    f"\\eta=\\Pi_{{Eff}}=\\ln(\\frac{{\\chi}}{{\\mu}})",
    # Utilization
    # f"\\rho=\\Pi_{{Util}}=\\ln(\\frac{{\\lambda}}{{\\mu}})",
    # # Stall
    # f"S=\\Pi_{{Stall}}=\\ln(\\frac{{\\lambda*W}}{{K}})",
]

# Verify columns exist
available_cols = [
    col for col in original_pi_cols if col in dflt_sys_coef.columns]
print(f"Available Original Π Coefficients: {len(available_cols)}/{len(original_pi_cols)}")

if len(available_cols) < len(original_pi_cols):
    print("⚠ WARNING: Some Π coefficients missing!")
    print(f"Found: {available_cols}")
else:
    print(f"✓ All {len(original_pi_cols)} original Π coefficients found")
    print(f"Available Coefficients: {available_cols}")

# Extract features (Occ, Eff) and target (Stall)
stall_col = f"S=\\Pi_{{Stall}}=\\ln(\\frac{{\\lambda*W}}{{K}})"
print(f"\nTarget (y): {stall_col}")

print(f"Features: {available_cols}")
X_full = np.log1p(dflt_sys_coef[available_cols].values)
y_full = np.log1p(dflt_sys_coef[stall_col].values)

# Clean data
mask = (
    np.isfinite(X_full).all(axis=1) &
    np.isfinite(y_full) &
    (y_full > 0) &
    (np.log1p(dflt_sys_coef[deriv_cols[0]]) < 1.0)  # &      # Occupancy < 1
    # (dflt_sys_coef[deriv_cols[1]] < 1.0)        # Efficiency < 1
)

X = X_full[mask]
y = y_full[mask]
print(X.shape, y.shape)

print(f"\nCleaned Data: {len(y)} samples")
for i in range(X.shape[1]):
    print(f"  Π{i+1}: [{X[:, i].min():.6f}, {X[:, i].max():.6f}]")

print(f"  Stall (S): [{y.min():.6f}, {y.max():.6f}]")

# ============================================================================
# MODEL 1: POLYNOMIAL COMBINATION OF Π COEFFICIENTS
# ============================================================================

print(f"\n{'='*80}")
print("MODEL 1: POLYNOMIAL (QUEUE-THEORY-INSPIRED)")
print(f"{'='*80}")
print("Formula: S = A·Occ + B·Eff² + C")


def polynomial_pi_model(X, A, B, C):
    """
    S = A·Π₁ + B·Π₂ + C·Π₃ + D·Π₄ + E·Π₅ + F·Π₁² + G·Π₅² + H
    
    Theory: 
    - Π₁ (λW/c) directly contributes to stall
    - Π₅ (L/c) shows queue occupancy, should have quadratic effect
    - Π₂, Π₃, Π₄ are modulating factors
    """
    pi_occ, pi_eff = X
    return (A*pi_occ + B*pi_eff**2 + C)


try:
    # Fit model
    X_T = X.T
    params_poly, cov_poly = curve_fit(
        polynomial_pi_model,
        X_T,
        y,
        p0=[1, -0.5, 0.5],
        maxfev=50000
    )

    # Predictions
    y_pred_poly = polynomial_pi_model(X_T, *params_poly)

    # Metrics
    r2_poly = r2_score(y, y_pred_poly)
    rmse_poly = np.sqrt(mean_squared_error(y, y_pred_poly))
    mae_poly = mean_absolute_error(y, y_pred_poly)

    # Cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    for train_idx, val_idx in kf.split(X):
        X_train = X[train_idx].T
        y_train = y[train_idx]
        X_val = X[val_idx].T
        y_val = y[val_idx]

        try:
            params_cv, _ = curve_fit(polynomial_pi_model,
                                     X_train,
                                     y_train,
                                     p0=[1, -0.5, 0.5],
                                     maxfev=10000)
            y_pred_cv = polynomial_pi_model(X_val, *params_cv)
            cv_scores.append(r2_score(y_val, y_pred_cv))
        except:
            continue

    cv_r2_mean = np.mean(cv_scores) if cv_scores else np.nan
    cv_r2_std = np.std(cv_scores) if cv_scores else np.nan

    # Print results
    print(f"\n--- Parameters ---")
    labels = ['A(Π_occ)', 'B(Π_eff)', 'C(Π_extra)']
    for label, param in zip(labels, params_poly):
        print(f"  {label:10s}: {param:10.6f}")

    print(f"\n--- Performance ---")
    print(f"  R²:              {r2_poly:.6f}")
    print(f"  RMSE:            {rmse_poly:.6f}")
    print(f"  MAE:             {mae_poly:.6f}")
    print(f"  CV-R² (5-fold):  {cv_r2_mean:.6f} ± {cv_r2_std:.6f}")

    print(f"\n✓ Successfully fitted Polynomial Model")

except Exception as e:
    print(f"\n✗ Failed: {str(e)}")
    r2_poly = np.nan

# ============================================================================
# MODEL 2: MACHINE LEARNING ENSEMBLE (XGBoost)
# ============================================================================

print(f"\n{'='*80}")
print("MODEL 2: XGBOOST REGRESSOR (Tree-Based Ensemble)")
print(f"{'='*80}")

try:
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # XGBoost model
    xgb_model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    xgb_model.fit(X_scaled, y)

    # Predictions
    y_pred_xgb = xgb_model.predict(X_scaled)

    # Metrics
    r2_xgb = r2_score(y, y_pred_xgb)
    rmse_xgb = np.sqrt(mean_squared_error(y, y_pred_xgb))
    mae_xgb = mean_absolute_error(y, y_pred_xgb)

    # Cross-validation
    cv_scores_xgb = cross_val_score(xgb_model, X_scaled, y, cv=5,
                                    scoring='r2')

    print(f"\n--- Feature Importance ---")
    for i, importance in enumerate(xgb_model.feature_importances_):
        print(f"  Π{i+1}: {importance:.4f}")

    print(f"\n--- Performance ---")
    print(f"  R²:              {r2_xgb:.6f}")
    print(f"  RMSE:            {rmse_xgb:.6f}")
    print(f"  MAE:             {mae_xgb:.6f}")
    print(
        f"  CV-R² (5-fold):  {cv_scores_xgb.mean():.6f} ± {cv_scores_xgb.std():.6f}")

    print(f"\n✓ Successfully trained XGBoost")

except Exception as e:
    print(f"\n✗ Failed: {str(e)}")
    r2_xgb = np.nan

# ============================================================================
# MODEL 3: NEURAL NETWORK (Simple MLP)
# ============================================================================

print(f"\n{'='*80}")
print("MODEL 3: NEURAL NETWORK (Multi-Layer Perceptron)")
print(f"{'='*80}")

try:
    from sklearn.neural_network import MLPRegressor

    # Standardize
    X_scaled = scaler.fit_transform(X)

    # MLP model
    mlp_model = MLPRegressor(
        hidden_layer_sizes=(64, 32, 16),
        activation='relu',
        solver='adam',
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42,
        early_stopping=True
    )

    mlp_model.fit(X_scaled, y)

    # Predictions
    y_pred_mlp = mlp_model.predict(X_scaled)

    # Metrics
    r2_mlp = r2_score(y, y_pred_mlp)
    rmse_mlp = np.sqrt(mean_squared_error(y, y_pred_mlp))
    mae_mlp = mean_absolute_error(y, y_pred_mlp)

    # Cross-validation
    cv_scores_mlp = cross_val_score(mlp_model, X_scaled, y, cv=5, scoring='r2')

    print(f"\n--- Performance ---")
    print(f"  R²:              {r2_mlp:.6f}")
    print(f"  RMSE:            {rmse_mlp:.6f}")
    print(f"  MAE:             {mae_mlp:.6f}")
    print(
        f"  CV-R² (5-fold):  {cv_scores_mlp.mean():.6f} ± {cv_scores_mlp.std():.6f}")

    print(f"\n✓ Successfully trained Neural Network")

except Exception as e:
    print(f"\n✗ Failed: {str(e)}")
    r2_mlp = np.nan

# ============================================================================
# COMPARISON TABLE
# ============================================================================

print(f"\n{'='*80}")
print("MODEL COMPARISON: ORIGINAL Π COEFFICIENTS")
print(f"{'='*80}\n")

comparison_results = pd.DataFrame({
    'Model': [
        'Polynomial (Queue Theory)',
        'XGBoost Ensemble',
        'Neural Network (MLP)',
        'Previous Best (Derived Vars)'
    ],
    'R²': [r2_poly, r2_xgb, r2_mlp, 0.002921],
    'RMSE': [rmse_poly if 'rmse_poly' in locals() else np.nan,
             rmse_xgb if 'rmse_xgb' in locals() else np.nan,
             rmse_mlp if 'rmse_mlp' in locals() else np.nan,
             3.624099],
    'Features': [
        'Π₁-Π₅ (polynomial)',
        'Π₁-Π₅ (tree ensemble)',
        'Π₁-Π₅ (neural net)',
        'O, η (derived)'
    ]
})

comparison_results = comparison_results.sort_values('R²', ascending=False)
print(comparison_results.to_string(index=False))

# Highlight best model
best_idx = comparison_results['R²'].idxmax()
best_model_name = comparison_results.loc[best_idx, 'Model']

print(f"\n{'='*80}")
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"{'='*80}")
print(f"   R² Score:  {comparison_results.loc[best_idx, 'R²']:.6f}")
print(f"   RMSE:      {comparison_results.loc[best_idx, 'RMSE']:.6f}")

# Calculate improvement
best_r2 = comparison_results.loc[best_idx, 'R²']
if best_r2 > 0.002921:
    improvement = ((best_r2 - 0.002921) / 0.002921) * 100
    print(f"\n📈 IMPROVEMENT: {improvement:.1f}% over previous best!")

# Save comparison
file_path_results = os.path.join(
    PATH, data_folder, results_folder, cs_folder, data_folder)
comparison_save_path = os.path.join(
    file_path_results, "original_pi_model_comparison.csv")
comparison_results.to_csv(comparison_save_path, index=False)
print(f"\n✓ Saved comparison to: {comparison_save_path}")

print(f"\n{'='*80}")
print("ANALYSIS COMPLETE: USING ORIGINAL Π COEFFICIENTS")
print(f"{'='*80}\n")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("--- Plotting Histograms for Derived Coefficients ---")

# Get all columns to plot
columns = dflt_sys_coef.columns.tolist()[0:9]
n_cols = len(columns)

# Calculate grid dimensions: 3 columns, n rows
n_columns = 3
n_rows = int(np.ceil(n_cols / n_columns))

# Create figure with subplots
fig, axes = plt.subplots(n_rows, n_columns, figsize=(18, 5 * n_rows))

# Flatten axes array for easier iteration
axes_flat = axes.flatten() if n_rows > 1 else [
    axes] if n_columns == 1 else axes

# Plot histogram for each coefficient
for idx, col in enumerate(columns):
    ax = axes_flat[idx]

    print(f"Plotting Histogram {idx+1}/{n_cols}: {col}")

    # Plot histogram with KDE
    try:
        # Remove invalid values (inf, nan)
        data = dflt_sys_coef[col].replace([np.inf, -np.inf], np.nan).dropna()

        if len(data) > 0:
            # Apply log transformation
            # Add small constant to avoid log(0)
            # log_data = np.log(data + 1e-10)
            log_data = np.log1p(data)

            # Plot histogram
            sns.histplot(log_data, bins=50, kde=True, ax=ax)
            ax.set_title(f"Histogram: ${col}$", fontsize=10, wrap=True)
            ax.set_xlabel("Log Value", fontsize=8)
            ax.set_ylabel("Frequency", fontsize=8)
            ax.grid(True, alpha=0.3)
            ax.tick_params(labelsize=7)
        else:
            ax.text(0.5, 0.5, 'No valid data',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f"Histogram: ${col}$", fontsize=10)

    except Exception as e:
        print(f"  ⚠ Error plotting {col}: {str(e)}")
        ax.text(0.5, 0.5, f'Error: {str(e)}',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f"Histogram: ${col}$", fontsize=10)

# Hide unused subplots
for idx in range(n_cols, len(axes_flat)):
    axes_flat[idx].set_visible(False)

# Adjust layout to prevent overlap
plt.tight_layout()

# Save figure
file_path_img = os.path.join(
    PATH, data_folder, results_folder, cs_folder, img_folder)
os.makedirs(file_path_img, exist_ok=True)
save_path = os.path.join(
    file_path_img, "dflt_coefficients_histograms_grid.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved histogram grid to: {save_path}")

# Display the plot
plt.show()

print(
    f"\n--- Finished Plotting {n_cols} Histograms in {n_rows}x{n_columns} Grid ---")

In [ ]:
# ...existing code...

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.optimize import curve_fit
import warnings
print("=" * 80)
print("QUEUE-THEORY-BASED REGRESSION MODEL")
print("Testing: S (Stall) ~ O / (1 - η)")
print("=" * 80)


warnings.filterwarnings('ignore')

# ============================================================================
# PREPARE DATA WITH YOUR HYPOTHESIS
# ============================================================================

print(f"\n{'='*80}")
print("DATA PREPARATION")
print(f"{'='*80}\n")

# Extract columns
occ_col = deriv_cols[0]      # Occupancy: O
stall_col = deriv_cols[1]    # Stall: S
eff_col = deriv_cols[2]      # Efficiency: η

# Create DataFrame from plot_df
data_df = pd.DataFrame(dflt_sys_coef[deriv_cols])

# Clean data: O ≤ 1.0 and η < 1.0 (must be < 1 for stability)
data_df = data_df[
    (data_df[occ_col] <= 1.0) &
    (data_df[eff_col] < 1.0) &
    (data_df[eff_col] > 0.0)
]

print(f"Cleaned Data: {len(data_df)} samples")
print(
    f"  O (Occupancy):  [{data_df[occ_col].min():.6f}, {data_df[occ_col].max():.6f}]")
print(
    f"  η (Efficiency): [{data_df[eff_col].min():.6f}, {data_df[eff_col].max():.6f}]")
print(
    f"  S (Stall):      [{data_df[stall_col].min():.6f}, {data_df[stall_col].max():.6f}]")

# ============================================================================
# MODEL 1: QUEUE-THEORY MODEL - S = A * [O / (1 - η)]
# ============================================================================

print(f"\n{'='*80}")
print("MODEL 1: QUEUE-THEORY MODEL (Your Hypothesis)")
print(f"{'='*80}")
print("Formula: S = A · [O / (1 - η)] + B")


def queue_theory_model_v1(X, A, B):
    """
    Queue-Theory Model: S = A * [O / (1 - η)] + B
    
    Theory: Stall grows as O/(1-η) when system approaches saturation
    (since η ≈ ρ and queue length ∝ 1/(1-ρ))
    
    Args:
        X: tuple of (O, η)
        A: scale factor
        B: offset
    """
    O, eta = X

    # Safety: prevent division by zero and ensure η < 1
    eta_safe = np.clip(eta, 0, 0.999)
    O_safe = np.maximum(O, 1e-10)

    # Queue-theory term: O/(1-η)
    queue_term = O_safe / (1 - eta_safe)

    return A * queue_term + B


# Prepare data for fitting
X = np.vstack([np.log1p(data_df[occ_col].values),
               np.log1p(data_df[eff_col].values)])
y = np.log1p(data_df[stall_col].values)

# Remove any remaining invalid values
mask = np.isfinite(X).all(axis=0) & np.isfinite(y) & (y > 0)
X = X[:, mask]
y = y[mask]

print(f"\nValid samples for fitting: {len(y)}")

try:
    # Fit model
    params_qt, cov_qt = curve_fit(
        queue_theory_model_v1,
        X,
        y,
        p0=[1.0, 0.1],  # Initial guess: A=1.0, B=0.1
        bounds=([0, -np.inf], [np.inf, np.inf]),  # A > 0
        maxfev=50000
    )

    # Predictions
    y_pred_qt = queue_theory_model_v1(X, *params_qt)

    # Basic metrics
    r2_qt = r2_score(y, y_pred_qt)
    rmse_qt = np.sqrt(mean_squared_error(y, y_pred_qt))
    mae_qt = mean_absolute_error(y, y_pred_qt)

    # Residuals
    residuals_qt = y - y_pred_qt
    residual_std_qt = np.std(residuals_qt)

    # Cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores_qt = []

    for train_idx, val_idx in kf.split(X.T):
        X_train = X[:, train_idx]
        y_train = y[train_idx]
        X_val = X[:, val_idx]
        y_val = y[val_idx]

        try:
            params_cv, _ = curve_fit(
                queue_theory_model_v1,
                X_train,
                y_train,
                p0=[1.0, 0.1],
                bounds=([0, -np.inf], [np.inf, np.inf]),
                maxfev=10000
            )
            y_pred_cv = queue_theory_model_v1(X_val, *params_cv)
            cv_scores_qt.append(r2_score(y_val, y_pred_cv))
        except:
            continue

    cv_r2_mean_qt = np.mean(cv_scores_qt) if cv_scores_qt else np.nan
    cv_r2_std_qt = np.std(cv_scores_qt) if cv_scores_qt else np.nan

    # AIC and BIC
    n = len(y)
    k = len(params_qt)
    rss = np.sum(residuals_qt**2)
    aic_qt = n * np.log(rss/n) + 2*k
    bic_qt = n * np.log(rss/n) + k*np.log(n)

    # Print results
    print(f"\n--- Parameters ---")
    print(f"  A (Scale):    {params_qt[0]:.6f}")
    print(f"  B (Offset):   {params_qt[1]:.6f}")

    print(f"\n--- Performance Metrics ---")
    print(f"  R²:              {r2_qt:.6f}")
    print(f"  RMSE:            {rmse_qt:.6f}")
    print(f"  MAE:             {mae_qt:.6f}")
    print(f"  CV-R² (5-fold):  {cv_r2_mean_qt:.6f} ± {cv_r2_std_qt:.6f}")
    print(f"  AIC:             {aic_qt:.2f}")
    print(f"  BIC:             {bic_qt:.2f}")
    print(f"  Residual Std:    {residual_std_qt:.6f}")

    # Interpretation
    print(f"\n--- Model Interpretation ---")
    print(f"  Stall = {params_qt[0]:.4f} · [O/(1-η)] + {params_qt[1]:.4f}")
    print(f"  → When η → 1 (saturation), stall grows exponentially ✓")
    print(f"  → When O increases, stall increases linearly ✓")

    print(f"\n✓ Successfully fitted Queue-Theory Model")

except Exception as e:
    print(f"\n✗ Failed to fit model: {str(e)}")
    r2_qt = np.nan

# ============================================================================
# MODEL 2: EXTENDED QUEUE-THEORY MODEL WITH LOG TRANSFORM
# ============================================================================

print(f"\n{'='*80}")
print("MODEL 2: LOG-TRANSFORMED QUEUE-THEORY MODEL")
print(f"{'='*80}")
print("Formula: log(S) = A · log[O / (1 - η)] + B")


def queue_theory_model_v2(X, A, B):
    """
    Log-transformed: log(S) = A * log[O/(1-η)] + B
    => S = exp(B) * [O/(1-η)]^A
    """
    O, eta = X

    eta_safe = np.clip(eta, 0, 0.999)
    O_safe = np.maximum(O, 1e-10)

    queue_term = O_safe / (1 - eta_safe)
    queue_term_safe = np.maximum(queue_term, 1e-10)

    # Log-space model
    log_S = A * np.log(queue_term_safe) + B
    return np.exp(log_S)


try:
    params_qt2, cov_qt2 = curve_fit(
        queue_theory_model_v2,
        X,
        y,
        p0=[1.0, 0.0],
        bounds=([-5, -np.inf], [5, np.inf]),
        maxfev=50000
    )

    y_pred_qt2 = queue_theory_model_v2(X, *params_qt2)

    r2_qt2 = r2_score(y, y_pred_qt2)
    rmse_qt2 = np.sqrt(mean_squared_error(y, y_pred_qt2))
    mae_qt2 = mean_absolute_error(y, y_pred_qt2)

    print(f"\n--- Parameters ---")
    print(f"  A (Power):     {params_qt2[0]:.6f}")
    print(f"  B (Log Scale): {params_qt2[1]:.6f}")

    print(f"\n--- Performance Metrics ---")
    print(f"  R²:              {r2_qt2:.6f}")
    print(f"  RMSE:            {rmse_qt2:.6f}")
    print(f"  MAE:             {mae_qt2:.6f}")

    print(f"\n✓ Successfully fitted Log-Transformed Queue-Theory Model")

except Exception as e:
    print(f"\n✗ Failed to fit model: {str(e)}")
    r2_qt2 = np.nan

# ============================================================================
# COMPARISON WITH PREVIOUS BEST MODEL
# ============================================================================

print(f"\n{'='*80}")
print("MODEL COMPARISON")
print(f"{'='*80}\n")

comparison_results = pd.DataFrame({
    'Model': [
        'Queue-Theory: S = A·[O/(1-η)] + B',
        'Log Queue-Theory: S = exp(B)·[O/(1-η)]^A',
        'Previous Best (Power-Law)',  # From your earlier results
    ],
    'R²': [r2_qt, r2_qt2, 0.002921],  # Your best previous R² was 0.002921
    'RMSE': [rmse_qt, rmse_qt2, 3.624099],
    'Formula': [
        f'{params_qt[0]:.3f}·[O/(1-η)] + {params_qt[1]:.3f}',
        f'exp({params_qt2[1]:.3f})·[O/(1-η)]^{params_qt2[0]:.3f}',
        'A·O^B·η^C'
    ]
})

comparison_results = comparison_results.sort_values('R²', ascending=False)
print(comparison_results.to_string(index=False))

# Highlight best model
best_idx = comparison_results['R²'].idxmax()
best_name = comparison_results.loc[best_idx, 'Model']

print(f"\n{'='*80}")
print(f"🏆 BEST MODEL: {best_name}")
print(f"{'='*80}")
print(f"   R² Score:  {comparison_results.loc[best_idx, 'R²']:.6f}")
print(f"   RMSE:      {comparison_results.loc[best_idx, 'RMSE']:.6f}")
print(f"   Formula:   {comparison_results.loc[best_idx, 'Formula']}")

# Calculate improvement
if r2_qt > 0.002921:
    improvement = ((r2_qt - 0.002921) / 0.002921) * 100
    print(
        f"\n📈 IMPROVEMENT: {improvement:.1f}% increase in R² over previous best!")

# Save results
file_path_results = os.path.join(
    PATH, data_folder, results_folder, cs_folder, data_folder)
comparison_save_path = os.path.join(
    file_path_results, "queue_theory_model_comparison.csv")
comparison_results.to_csv(comparison_save_path, index=False)
print(f"\n✓ Saved comparison to: {comparison_save_path}")

print(f"\n{'='*80}")
print("QUEUE-THEORY REGRESSION ANALYSIS COMPLETE")
print(f"{'='*80}\n")

In [ ]:
import warnings
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, LogisticRegression
from scipy.optimize import curve_fit
from sklearn.preprocessing import PolynomialFeatures
print("=" * 80)
print("COMPREHENSIVE REGRESSION MODELS FOR STALL PREDICTION")
print("Testing: S (Stall) ~ f(O, η) using Multiple Functional Forms")
print("Models: Linear, Log-Linear, Weibull, Exponential, Gamma, Power-Law, Logistic")
print("=" * 80)

warnings.filterwarnings('ignore')

# Prepare data from plot_df dflt_sys_coef
occ_col = deriv_cols[0]      # Occupancy: O
stall_col = deriv_cols[1]    # Stall: S
eff_col = deriv_cols[2]      # Efficiency: η
data_df = pd.DataFrame(dflt_sys_coef[deriv_cols])
# remove occupancy > 1
data_df = data_df[data_df[occ_col] <= 1.0]

# Extract features and target
X = data_df[[occ_col, eff_col]].values
y = data_df[stall_col].values

# Remove invalid values
mask = np.isfinite(X).all(axis=1) & np.isfinite(y) & (y > 0)
X = X[mask]
y = y[mask]

print(f"\nData: {len(y)} valid samples")
print(f"Occupancy (O): [{X[:, 0].min():.4f}, {X[:, 0].max():.4f}]")
print(f"Efficiency (η): [{X[:, 1].min():.4f}, {X[:, 1].max():.4f}]")
print(f"Stall (S): [{y.min():.4f}, {y.max():.4f}]")

# ============================================================================
# DEFINE ALL REGRESSION MODELS
# ============================================================================

# 1. Linear Model: S = β₀ + β₁·O + β₂·η


def linear_model(X, beta0, beta1, beta2):
    """Linear: S = β₀ + β₁·O + β₂·η"""
    O, eta = X
    return beta0 + beta1 * O + beta2 * eta


# 2. Log-Linear Model: log(S) = β₀ + β₁·O + β₂·η
def log_linear_predict(X, beta0, beta1, beta2):
    """Log-Linear: S = exp(β₀ + β₁·O + β₂·η)"""
    O, eta = X
    return np.exp(beta0 + beta1 * O + beta2 * eta)


# 3. Power-Law Model: S = A · O^B · η^C
def power_law_model(X, A, B, C):
    """Power-Law: S = A · O^B · η^C"""
    O, eta = X
    O_safe = np.maximum(O, 1e-10)
    eta_safe = np.maximum(np.abs(eta), 1e-10)
    return A * np.power(O_safe, B) * np.power(eta_safe, C)


# 4. Exponential Model: S = A · exp(B·O + C·η)
def exponential_model(X, A, B, C):
    """Exponential: S = A · exp(B·O + C·η)"""
    O, eta = X
    exponent = B * O + C * eta
    exponent = np.clip(exponent, -50, 50)  # Prevent overflow
    return A * np.exp(exponent)


# 5. Weibull-like Model: S = A · [1 - exp(-(B·O + C·η)^k)]
def weibull_model(X, A, B, C, k):
    """Weibull-like: S = A · [1 - exp(-(B·O + C·η)^k)]"""
    O, eta = X
    linear_part = B * O + C * eta
    linear_part = np.maximum(linear_part, 0)
    power_part = np.power(linear_part, k)
    power_part = np.clip(power_part, 0, 50)  # Prevent overflow
    return A * (1 - np.exp(-power_part))


# 6. Gamma-like Model: S = A · (B·O + C·η + ε)^k
def gamma_model(X, A, B, C, k):
    """Gamma-like: S = A · (B·O + C·η + ε)^k"""
    O, eta = X
    linear_part = B * O + C * eta + 0.01
    linear_part = np.maximum(linear_part, 0.01)
    return A * np.power(linear_part, k)


# 7. Logistic Model: S = L / (1 + exp(-k·(B·O + C·η - x₀)))
def logistic_model(X, L, k, B, C, x0):
    """Logistic: S = L / (1 + exp(-k·(B·O + C·η - x₀)))"""
    O, eta = X
    z = k * (B * O + C * eta - x0)
    z = np.clip(z, -50, 50)  # Prevent overflow
    return L / (1 + np.exp(-z))


# ============================================================================
# FIT ALL MODELS WITH CROSS-VALIDATION
# ============================================================================

models_config = {
    'Linear': {
        'func': linear_model,
        'p0': [0.1, 1.0, -0.5],
        'bounds': ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])
    },
    'Log-Linear': {
        'func': log_linear_predict,
        'p0': [0.0, 1.0, -1.0],
        'bounds': ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])
    },
    'Power-Law': {
        'func': power_law_model,
        'p0': [1.0, 1.0, -0.5],
        'bounds': ([0, -5, -5], [np.inf, 5, 5])
    },
    'Exponential': {
        'func': exponential_model,
        'p0': [1.0, 1.0, -1.0],
        'bounds': ([0, -10, -10], [np.inf, 10, 10])
    },
    'Weibull': {
        'func': weibull_model,
        'p0': [10.0, 0.5, 0.5, 1.5],
        'bounds': ([0, 0, -10, 0.1], [np.inf, 10, 10, 5])
    },
    'Gamma': {
        'func': gamma_model,
        'p0': [1.0, 1.0, -0.5, 2.0],
        'bounds': ([0, -10, -10, 0.1], [np.inf, 10, 10, 10])
    },
    'Logistic': {
        'func': logistic_model,
        'p0': [y.max(), 1.0, 1.0, -0.5, 0.5],
        'bounds': ([0, 0.01, -10, -10, -10], [np.inf, 10, 10, 10, 10])
    }
}

results_summary = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for model_name, config in models_config.items():
    print(f"\n{'='*60}")
    print(f"MODEL: {model_name}")
    print(f"{'='*60}")

    try:
        # Transpose for curve_fit (expects rows as variables)
        X_T = X.T

        # Fit model
        params, cov = curve_fit(
            config['func'],
            X_T,
            y,
            p0=config['p0'],
            bounds=config['bounds'],
            maxfev=50000,
            method='trf'
        )

        # Predictions
        y_pred = config['func'](X_T, *params)

        # Metrics
        r2 = r2_score(y, y_pred)
        rmse = np.sqrt(mean_squared_error(y, y_pred))
        mae = mean_absolute_error(y, y_pred)

        # Residuals
        residuals = y - y_pred
        residual_std = np.std(residuals)

        # Cross-validation
        cv_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train = X[train_idx].T
            y_train = y[train_idx]
            X_val = X[val_idx].T
            y_val = y[val_idx]

            try:
                params_cv, _ = curve_fit(
                    config['func'],
                    X_train,
                    y_train,
                    p0=config['p0'],
                    bounds=config['bounds'],
                    maxfev=10000
                )
                y_pred_cv = config['func'](X_val, *params_cv)
                cv_scores.append(r2_score(y_val, y_pred_cv))
            except:
                continue

        cv_r2_mean = np.mean(cv_scores) if cv_scores else np.nan
        cv_r2_std = np.std(cv_scores) if cv_scores else np.nan

        # AIC and BIC
        n = len(y)
        k = len(params)
        rss = np.sum(residuals**2)
        aic = n * np.log(rss/n) + 2*k
        bic = n * np.log(rss/n) + k*np.log(n)

        # Print results
        print(f"\nParameters: {params}")
        print(f"\nPerformance:")
        print(f"  R²:              {r2:.6f}")
        print(f"  RMSE:            {rmse:.6f}")
        print(f"  MAE:             {mae:.6f}")
        print(f"  CV-R² (5-fold):  {cv_r2_mean:.6f} ± {cv_r2_std:.6f}")
        print(f"  AIC:             {aic:.2f}")
        print(f"  BIC:             {bic:.2f}")
        print(f"  Residual Std:    {residual_std:.6f}")

        # Store results
        results_summary.append({
            'Model': model_name,
            'R²': r2,
            'RMSE': rmse,
            'MAE': mae,
            'CV-R²': cv_r2_mean,
            'CV-Std': cv_r2_std,
            'AIC': aic,
            'BIC': bic,
            'Res_Std': residual_std,
            'Params': len(params)
        })

        print(f"\n✓ Successfully fitted {model_name}")

    except Exception as e:
        print(f"\n✗ Failed to fit {model_name}: {str(e)}")
        results_summary.append({
            'Model': model_name,
            'R²': np.nan,
            'RMSE': np.nan,
            'MAE': np.nan,
            'CV-R²': np.nan,
            'CV-Std': np.nan,
            'AIC': np.nan,
            'BIC': np.nan,
            'Res_Std': np.nan,
            'Params': len(config['p0'])
        })

# ============================================================================
# COMPREHENSIVE COMPARISON TABLE
# ============================================================================
print(f"\n{'='*80}")
print("COMPREHENSIVE MODEL COMPARISON")
print(f"{'='*80}\n")

comparison_df = pd.DataFrame(results_summary)
comparison_df = comparison_df.sort_values('R²', ascending=False)

print(comparison_df.to_string(index=False))

# Select best models
best_r2_idx = comparison_df['R²'].idxmax()
best_aic_idx = comparison_df['AIC'].idxmin()

print(f"\n{'='*80}")
print(f"🏆 BEST MODEL BY R²: {comparison_df.loc[best_r2_idx, 'Model']}")
print(f"{'='*80}")
print(f"   R² Score:        {comparison_df.loc[best_r2_idx, 'R²']:.6f}")
print(f"   RMSE:            {comparison_df.loc[best_r2_idx, 'RMSE']:.6f}")
print(f"   CV-R²:           {comparison_df.loc[best_r2_idx, 'CV-R²']:.6f}")

print(f"\n{'='*80}")
print(f"🏆 BEST MODEL BY AIC: {comparison_df.loc[best_aic_idx, 'Model']}")
print(f"{'='*80}")
print(f"   AIC:             {comparison_df.loc[best_aic_idx, 'AIC']:.2f}")
print(f"   R²:              {comparison_df.loc[best_aic_idx, 'R²']:.6f}")
print(f"   RMSE:            {comparison_df.loc[best_aic_idx, 'RMSE']:.6f}")

# Save comparison
file_path_results = os.path.join(
    PATH, data_folder, results_folder, cs_folder, data_folder)
comparison_save_path = os.path.join(
    file_path_results, "comprehensive_regression_comparison.csv")
comparison_df.to_csv(comparison_save_path, index=False)
print(f"\n✓ Saved comprehensive comparison to: {comparison_save_path}")

print(f"\n{'='*80}")
print("COMPREHENSIVE REGRESSION ANALYSIS COMPLETE")
print(f"{'='*80}\n")

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error  # ADD THIS LINE
import warnings
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
import numpy as np  # ADD THIS IF NOT ALREADY IMPORTED
import pandas as pd  # ADD THIS IF NOT ALREADY IMPORTED

print("="*80)
print("REGRESSION MODELS FOR STALL PREDICTION")
print("="*80)

warnings.filterwarnings('ignore')

# ...existing code...

# Prepare data
occ_col = deriv_cols[0]      # Occupancy
stall_col = deriv_cols[1]    # Stall
eff_col = deriv_cols[2]      # Efficiency

# Extract features and target
X = plot_df[[occ_col, eff_col]].values
y = plot_df[stall_col].values

# Remove invalid values
mask = np.isfinite(X).all(axis=1) & np.isfinite(y) & (y > 0)
X = X[mask]
y = y[mask]

print(f"\nData: {len(y)} valid samples")
print(f"Occupancy (O): [{X[:, 0].min():.4f}, {X[:, 0].max():.4f}]")
print(f"Efficiency (η): [{X[:, 1].min():.4f}, {X[:, 1].max():.4f}]")
print(f"Stall (S): [{y.min():.4f}, {y.max():.4f}]")

# ============================================================================
# MODEL 1: LOG-TRANSFORMED LINEAR REGRESSION (Best for skewed data)
# ============================================================================
print("\n" + "="*60)
print("MODEL 1: LOG-TRANSFORMED LINEAR REGRESSION")
print("="*60)
print("Formula: log(S) = β₀ + β₁·log(O) + β₂·log(η)")


# Apply log transformation
X_log = np.log1p(X)  # log(1 + x) to handle zeros
y_log = np.log1p(y)

# Fit model
lr_log = LinearRegression()
lr_log.fit(X_log, y_log)

# Cross-validation
cv_scores = cross_val_score(lr_log, X_log, y_log, cv=5,
                            scoring='neg_mean_squared_error')
rmse_cv = np.sqrt(-cv_scores.mean())

# Predictions in original scale
y_pred_log = np.expm1(lr_log.predict(X_log))  # inverse: exp(x) - 1
r2_log = r2_score(y, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y, y_pred_log))

print(f"\nCoefficients:")
print(f"  β₀ (Intercept): {lr_log.intercept_:.4f}")
print(f"  β₁ (Occupancy): {lr_log.coef_[0]:.4f}")
print(f"  β₂ (Efficiency): {lr_log.coef_[1]:.4f}")
print(f"\nPerformance:")
print(f"  R²:              {r2_log:.6f}")
print(f"  RMSE:            {rmse_log:.6f}")
print(f"  CV-RMSE (5-fold): {rmse_cv:.6f}")

# ============================================================================
# MODEL 2: POWER-LAW REGRESSION (From your previous analysis)
# ============================================================================
print("\n" + "="*60)
print("MODEL 2: POWER-LAW REGRESSION")
print("="*60)
print("Formula: S = A · O^B · η^C")


def power_law_model(X, A, B, C):
    """S = A * O^B * η^C"""
    O, eta = X
    O_safe = np.maximum(O, 1e-10)
    eta_safe = np.maximum(eta, 1e-10)
    return A * np.power(O_safe, B) * np.power(eta_safe, C)


try:
    X_T = X.T  # Transpose for curve_fit
    params_pl, _ = curve_fit(
        power_law_model,
        X_T,
        y,
        p0=[1.0, 1.0, -0.5],
        bounds=([0, -5, -5], [np.inf, 5, 5]),
        maxfev=10000
    )

    y_pred_pl = power_law_model(X_T, *params_pl)
    r2_pl = r2_score(y, y_pred_pl)
    rmse_pl = np.sqrt(mean_squared_error(y, y_pred_pl))

    print(f"\nParameters:")
    print(f"  A (Scale):       {params_pl[0]:.4f}")
    print(f"  B (Occ Power):   {params_pl[1]:.4f}")
    print(f"  C (Eff Power):   {params_pl[2]:.4f}")
    print(f"\nPerformance:")
    print(f"  R²:              {r2_pl:.6f}")
    print(f"  RMSE:            {rmse_pl:.6f}")

except Exception as e:
    print(f"  ✗ Failed: {str(e)}")
    r2_pl = 0
    rmse_pl = np.inf

# ============================================================================
# MODEL 3: GRADIENT BOOSTING (Non-parametric, best for complex patterns)
# ============================================================================
print("\n" + "="*60)
print("MODEL 3: GRADIENT BOOSTING REGRESSOR")
print("="*60)

# Apply Yeo-Johnson transformation (handles negatives better than log)
transformer = PowerTransformer(method='yeo-johnson')
X_transformed = transformer.fit_transform(X)

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

gb_model.fit(X_transformed, y)

# Cross-validation
cv_scores_gb = cross_val_score(gb_model, X_transformed, y, cv=5,
                               scoring='neg_mean_squared_error')
rmse_cv_gb = np.sqrt(-cv_scores_gb.mean())

y_pred_gb = gb_model.predict(X_transformed)
r2_gb = r2_score(y, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y, y_pred_gb))

print(f"\nFeature Importance:")
for i, feat_name in enumerate(['Occupancy', 'Efficiency']):
    print(f"  {feat_name}: {gb_model.feature_importances_[i]:.4f}")
print(f"\nPerformance:")
print(f"  R²:              {r2_gb:.6f}")
print(f"  RMSE:            {rmse_gb:.6f}")
print(f"  CV-RMSE (5-fold): {rmse_cv_gb:.6f}")

# ============================================================================
# MODEL 4: GENERALIZED ADDITIVE MODEL (GAM) - Flexible non-linear
# ============================================================================
print("\n" + "="*60)
print("MODEL 4: POLYNOMIAL REGRESSION (2nd Order)")
print("="*60)
print("Formula: S = β₀ + β₁·O + β₂·η + β₃·O² + β₄·η² + β₅·O·η")


poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(X)

lr_poly = LinearRegression()
lr_poly.fit(X_poly, y)

cv_scores_poly = cross_val_score(lr_poly, X_poly, y, cv=5,
                                 scoring='neg_mean_squared_error')
rmse_cv_poly = np.sqrt(-cv_scores_poly.mean())

y_pred_poly = lr_poly.predict(X_poly)
r2_poly = r2_score(y, y_pred_poly)
rmse_poly = np.sqrt(mean_squared_error(y, y_pred_poly))

feature_names = poly.get_feature_names_out(['O', 'η'])
print(f"\nCoefficients:")
for name, coef in zip(feature_names, lr_poly.coef_):
    print(f"  {name}: {coef:.4f}")
print(f"  Intercept: {lr_poly.intercept_:.4f}")
print(f"\nPerformance:")
print(f"  R²:              {r2_poly:.6f}")
print(f"  RMSE:            {rmse_poly:.6f}")
print(f"  CV-RMSE (5-fold): {rmse_cv_poly:.6f}")

# ============================================================================
# COMPARISON TABLE
# ============================================================================
print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

comparison = pd.DataFrame({
    'Model': [
        'Log-Transformed Linear',
        'Power-Law',
        'Gradient Boosting',
        'Polynomial (2nd Order)'
    ],
    'R²': [r2_log, r2_pl, r2_gb, r2_poly],
    'RMSE': [rmse_log, rmse_pl, rmse_gb, rmse_poly],
    'CV-RMSE': [rmse_cv, np.nan, rmse_cv_gb, rmse_cv_poly],
    'Interpretability': ['High', 'High', 'Low', 'Medium'],
    'Extrapolation': ['Good', 'Good', 'Poor', 'Medium']
})

comparison = comparison.sort_values('R²', ascending=False)
print(comparison.to_string(index=False))

# Select best model
best_idx = comparison['R²'].idxmax()
best_model_name = comparison.loc[best_idx, 'Model']

print(f"\n🏆 RECOMMENDED MODEL: {best_model_name}")
print(f"   R² = {comparison.loc[best_idx, 'R²']:.6f}")
print(f"   RMSE = {comparison.loc[best_idx, 'RMSE']:.6f}")

# Save comparison
file_path_results = os.path.join(
    PATH, data_folder, results_folder, cs_folder, data_folder)
comparison_save_path = os.path.join(
    file_path_results, "stall_regression_comparison.csv")
comparison.to_csv(comparison_save_path, index=False)
print(f"\n✓ Saved comparison to: {comparison_save_path}")

In [ ]:
print("=" * 80)
print("COMPREHENSIVE STATISTICAL MODEL VALIDATION")
print("Testing: S (Stall) ~ f(O, η) for Multiple Probability Distributions")
print("Including: K-S, Anderson-Darling, and Chi-Square Tests")
print("=" * 80)

# Import additional statistical libraries
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Helper function for Runs Test (MUST BE DEFINED BEFORE USE)
def _runs_test(residuals):
    """Wald-Wolfowitz runs test for randomness"""
    median = np.median(residuals)
    runs = np.diff(residuals > median).sum() + 1
    n1 = np.sum(residuals > median)
    n2 = np.sum(residuals <= median)
    
    if n1 == 0 or n2 == 0:
        return 0, 1.0
    
    runs_exp = ((2 * n1 * n2) / (n1 + n2)) + 1
    runs_var = (2 * n1 * n2 * (2 * n1 * n2 - n1 - n2)) / \
               ((n1 + n2)**2 * (n1 + n2 - 1))
    
    if runs_var == 0:
        return 0, 1.0
        
    z_stat = (runs - runs_exp) / np.sqrt(runs_var)
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    return z_stat, p_value

# Prepare data from plot_df
print(f"\n{'='*80}")
print("DATA PREPARATION FROM plot_df")
print(f"{'='*80}\n")

# Extract columns
occ_col = deriv_cols[0]      # Occupancy: O
stall_col = deriv_cols[1]    # Stall: S
eff_col = deriv_cols[2]      # Efficiency: η
util_col = deriv_cols[3]     # Utilization: ρ (ADDED)

print(f"Independent Variables:")
print(f"  X1 (Occupancy):   {occ_col}")
print(f"  X2 (Efficiency):  {eff_col}")
print(f"  X3 (Utilization): {util_col}")  # ADDED
print(f"Dependent Variable:")
print(f"  Y (Stall):        {stall_col}")

# Extract clean data
O = plot_df[occ_col].values      # Occupancy
S = plot_df[stall_col].values    # Stall
eta = plot_df[eff_col].values    # Efficiency
rho = plot_df[util_col].values   # Utilization (ADDED)

# Remove any invalid values
mask = (S > 0) & np.isfinite(S) & np.isfinite(O) & np.isfinite(eta) & np.isfinite(rho)  # UPDATED
X_data = np.vstack([O[mask], eta[mask], rho[mask]])  # UPDATED: added rho
y_data = S[mask]

print(f"\nData Summary:")
print(f"  Valid data points:  {np.sum(mask)} / {len(S)} ({100*np.sum(mask)/len(S):.1f}%)")
print(f"  Occupancy (O):      [{O[mask].min():.6f}, {O[mask].max():.6f}]")
print(f"  Efficiency (η):     [{eta[mask].min():.6f}, {eta[mask].max():.6f}]")
print(f"  Utilization (ρ):    [{rho[mask].min():.6f}, {rho[mask].max():.6f}]")  # ADDED
print(f"  Stall (S):          [{y_data.min():.6f}, {y_data.max():.6f}]")

# ============================================================================
# STEP 1: Define Multiple Distribution Models
# ============================================================================
print(f"\n{'='*80}")
print("DEFINING PROBABILITY DISTRIBUTION MODELS")
print(f"{'='*80}\n")


# 1. Exponential Model
def exponential_model(X, A, B, C):
    """Exponential: S = A * exp(B*O + C*η)"""
    O, eta, rho = X  # UPDATED: unpack 3 variables
    return A * np.exp(B * O + C * eta)


# 2. Gamma-like Model
def gamma_model(X, A, B, C, k):
    """Gamma-inspired: S = A * (B*O + C*η + ε)^k"""
    O, eta, rho = X  # UPDATED
    linear_part = B * O + C * eta + 0.01
    linear_part = np.maximum(linear_part, 0.01)
    return A * np.power(linear_part, k)


# 3. Normal/Linear Model
def normal_model(X, A, B, C):
    """Normal/Linear: S = A + B*O + C*η"""
    O, eta, rho = X  # UPDATED
    return A + B * O + C * eta


# 4. Weibull-like Model
def weibull_model(X, A, B, C, k):
    """Weibull-inspired: S = A * (1 - exp(-(B*O + C*η)^k))"""
    O, eta, rho = X  # UPDATED
    linear_part = B * O + C * eta
    linear_part = np.maximum(linear_part, 0)
    return A * (1 - np.exp(-np.power(linear_part, k)))


# 5. Log-Normal Model
def lognormal_model(X, A, B, C):
    """Log-Normal: ln(S) = A + B*O + C*η => S = exp(A + B*O + C*η)"""
    O, eta, rho = X  # UPDATED
    return np.exp(A + B * O + C * eta)


# 6. Power Law Model
def power_model(X, A, B, C):
    """Power Law: S = A * O^B * η^C"""
    O, eta, rho = X  # UPDATED
    O_safe = np.maximum(O, 1e-10)
    eta_safe = np.maximum(np.abs(eta), 1e-10)
    return A * np.power(O_safe, B) * np.power(eta_safe, C)


# # 7. Analytical Queue Model: S = A * [-1/(η - ρ) * ln(O / (ρ * (η - ρ)))]
# def analytical_queue_model(X, A):
#     """
#     Analytical Queue Model based on queueing theory:
#     S = A * [-1/(η - ρ) * ln(O / (ρ * (η - ρ)))]
    
#     Where:
#     - O: Occupancy (X[0])
#     - η: Efficiency (X[1])
#     - ρ: Utilization (X[2])
#     - A: Scale factor (fitted parameter)
    
#     Args:
#         X: tuple of (O, η, ρ)
#         A: scale factor parameter
#     """
#     O, eta, rho = X

#     # Ensure positive values and avoid division by zero
#     O_safe = np.maximum(O, 1e-10)
#     eta_safe = np.maximum(eta, 1e-10)
#     rho_safe = np.clip(rho, 1e-10, 0.999)  # Keep ρ < 1

#     # Ensure η > ρ for stability
#     eta_safe = np.maximum(eta_safe, rho_safe + 1e-6)

#     # Compute denominator: (η - ρ)
#     denominator = eta_safe - rho_safe
#     denominator = np.maximum(denominator, 1e-10)

#     # Compute argument for logarithm: O / (ρ * (η - ρ))
#     log_arg = O_safe / (rho_safe * denominator)
#     log_arg = np.maximum(log_arg, 1e-10)

#     # Compute Stall: S = A * [-1/(η - ρ) * ln(O / (ρ * (η - ρ)))]
#     stall = A * (-1.0 / denominator) * np.log(log_arg)

#     return stall


models = {
    'Exponential': {
        'func': exponential_model,
        'p0': [1.0, 1.0, -1.0],
        'bounds': ([0, -np.inf, -np.inf], [np.inf, np.inf, np.inf])
    },
    'Gamma': {
        'func': gamma_model,
        'p0': [1.0, 1.0, -0.5, 2.0],
        'bounds': ([0, -np.inf, -np.inf, 0.1], [np.inf, np.inf, np.inf, 10])
    },
    'Normal': {
        'func': normal_model,
        'p0': [0.1, 1.0, -0.5],
        'bounds': ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])
    },
    'Weibull': {
        'func': weibull_model,
        'p0': [10.0, 0.5, 0.5, 1.5],
        'bounds': ([0, 0, -np.inf, 0.1], [np.inf, np.inf, np.inf, 5])
    },
    'Log-Normal': {
        'func': lognormal_model,
        'p0': [0.0, 1.0, -1.0],
        'bounds': ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])
    },
    'Power-Law': {
        'func': power_model,
        'p0': [1.0, 1.0, -0.5],
        'bounds': ([0, -5, -5], [np.inf, 5, 5])
    },
![1761400871971](image/CS-01C-TAS-DimensionalModelling/1761400871971.png)ame in enumerate(models.keys(), 1):
    print(f"  {i}. {name}")

# ============================================================================
# STEP 2: Fit All Models with Comprehensive Testing
# ============================================================================
print(f"\n{'='*80}")
print("FITTING AND VALIDATING MODELS WITH DISTRIBUTION TESTS")
print(f"{'='*80}\n")

results = {}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for model_name, model_config in models.items():
    print(f"\n{'='*60}")
    print(f"Testing: {model_name} Model")
    print(f"{'='*60}")
    
    try:
        # Full dataset fit
        params, cov = curve_fit(
            model_config['func'],
            X_data,
            y_data,
            p0=model_config['p0'],
            bounds=model_config['bounds'],
            maxfev=50000,
            method='trf'
        )
        
        # Predictions
        y_pred = model_config['func'](X_data, *params)
        
        # Calculate basic metrics
        r2 = r2_score(y_data, y_pred)
        rmse = np.sqrt(mean_squared_error(y_data, y_pred))
        mae = mean_absolute_error(y_data, y_pred)
        
        # Residuals
        residuals = y_data - y_pred
        residual_std = np.std(residuals)
        residuals_norm = (residuals - np.mean(residuals)) / residual_std if residual_std > 0 else residuals
        
        print(f"\n--- Basic Fit Metrics ---")
        print(f"  R² Score: {r2:.6f}")
        print(f"  RMSE:     {rmse:.6f}")
        print(f"  MAE:      {mae:.6f}")
        
        # ====================================================================
        # DISTRIBUTION GOODNESS-OF-FIT TESTS
        # ====================================================================
        print(f"\n--- Goodness-of-Fit Tests for Residuals ---")
        
        # 1. Kolmogorov-Smirnov Test (Normality of Residuals)
        print(f"\n1. KOLMOGOROV-SMIRNOV TEST")
        print(f"   H0: Residuals follow a normal distribution")
        ks_stat, ks_pvalue = stats.kstest(residuals_norm, 'norm')
        print(f"   Statistic: {ks_stat:.6f}")
        print(f"   P-value:   {ks_pvalue:.6e}")
        if ks_pvalue > 0.05:
            print(f"   Result:    ✓ PASS - Cannot reject normality (α=0.05)")
        else:
            print(f"   Result:    ✗ FAIL - Reject normality (α=0.05)")
        
        # 2. Anderson-Darling Test (Normality of Residuals)
        print(f"\n2. ANDERSON-DARLING TEST")
        print(f"   H0: Residuals follow a normal distribution")
        ad_result = stats.anderson(residuals_norm, dist='norm')
        print(f"   Statistic: {ad_result.statistic:.6f}")
        print(f"   Critical Values (significance levels):")
        for i, (cv, sl) in enumerate(zip(ad_result.critical_values, ad_result.significance_level)):
            status = "✓" if ad_result.statistic < cv else "✗"
            print(f"     {status} {sl:5.1f}%: {cv:.6f}")
        
        # Overall result at 5% significance
        ad_pass = ad_result.statistic < ad_result.critical_values[2]  # 5% level is index 2
        if ad_pass:
            print(f"   Result:    ✓ PASS - Cannot reject normality (α=0.05)")
        else:
            print(f"   Result:    ✗ FAIL - Reject normality (α=0.05)")
        
        # 3. Chi-Square Goodness-of-Fit Test
        print(f"\n3. CHI-SQUARE GOODNESS-OF-FIT TEST")
        print(f"   H0: Residuals follow the theoretical distribution")
        
        # Create histogram bins
        n_bins = min(50, int(np.sqrt(len(residuals_norm))))
        observed_freq, bin_edges = np.histogram(residuals_norm, bins=n_bins)
        
        # Calculate expected frequencies under normal distribution
        expected_freq = len(residuals_norm) * np.diff(
            stats.norm.cdf(bin_edges, loc=0, scale=1)
        )
        
        # Remove bins with very low expected frequencies
        mask_bins = expected_freq >= 5
        observed_freq = observed_freq[mask_bins]
        expected_freq = expected_freq[mask_bins]
        
        if len(observed_freq) > 1:
            chi2_stat = np.sum((observed_freq - expected_freq)**2 / expected_freq)
            dof = len(observed_freq) - 1 - 2  # -1 for constraint, -2 for estimated params
            dof = max(1, dof)
            chi2_pvalue = 1 - stats.chi2.cdf(chi2_stat, dof)
            
            print(f"   Statistic:          {chi2_stat:.6f}")
            print(f"   Degrees of Freedom: {dof}")
            print(f"   P-value:            {chi2_pvalue:.6e}")
            print(f"   Bins used:          {len(observed_freq)}/{n_bins}")
            
            if chi2_pvalue > 0.05:
                print(f"   Result:             ✓ PASS - Cannot reject fit (α=0.05)")
            else:
                print(f"   Result:             ✗ FAIL - Reject fit (α=0.05)")
        else:
            chi2_stat = np.nan
            chi2_pvalue = np.nan
            print(f"   Result:             ⚠ INSUFFICIENT DATA for reliable test")
        
        # 4. Shapiro-Wilk Test (Additional verification)
        print(f"\n4. SHAPIRO-WILK TEST (Verification)")
        print(f"   H0: Residuals follow a normal distribution")
        # Use subset if dataset is very large
        sample_size = min(5000, len(residuals_norm))
        sw_sample = np.random.choice(residuals_norm, size=sample_size, replace=False)
        sw_stat, sw_pvalue = stats.shapiro(sw_sample)
        print(f"   Statistic: {sw_stat:.6f} (sample n={sample_size})")
        print(f"   P-value:   {sw_pvalue:.6e}")
        if sw_pvalue > 0.05:
            print(f"   Result:    ✓ PASS - Cannot reject normality (α=0.05)")
        else:
            print(f"   Result:    ✗ FAIL - Reject normality (α=0.05)")
        
        # ====================================================================
        # MODEL ADEQUACY TESTS
        # ====================================================================
        print(f"\n--- Model Adequacy Tests ---")
        
        # 5. Durbin-Watson (Autocorrelation)
        from statsmodels.stats.stattools import durbin_watson
        dw_stat = durbin_watson(residuals)
        print(f"\n5. DURBIN-WATSON TEST (Autocorrelation)")
        print(f"   Statistic: {dw_stat:.6f}")
        print(f"   Interpretation: ", end="")
        if 1.5 < dw_stat < 2.5:
            print(f"✓ No significant autocorrelation")
        else:
            print(f"⚠ Possible autocorrelation detected")
        
        # 6. Runs Test (Randomness)
        print(f"\n6. RUNS TEST (Randomness)")
        runs_stat, runs_pvalue = _runs_test(residuals)
        print(f"   Statistic: {runs_stat:.6f}")
        print(f"   P-value:   {runs_pvalue:.6e}")
        if runs_pvalue > 0.05:
            print(f"   Result:    ✓ Residuals appear random")
        else:
            print(f"   Result:    ⚠ Residuals may not be random")
        
        # Cross-validation
        cv_r2_scores = []
        for train_idx, val_idx in kf.split(X_data.T):
            X_train = X_data[:, train_idx]
            y_train = y_data[train_idx]
            X_val = X_data[:, val_idx]
            y_val = y_data[val_idx]
            
            try:
                params_cv, _ = curve_fit(
                    model_config['func'],
                    X_train,
                    y_train,
                    p0=model_config['p0'],
                    bounds=model_config['bounds'],
                    maxfev=10000
                )
                y_pred_cv = model_config['func'](X_val, *params_cv)
                cv_r2_scores.append(r2_score(y_val, y_pred_cv))
            except:
                continue
        
        cv_r2_mean = np.mean(cv_r2_scores) if cv_r2_scores else 0
        cv_r2_std = np.std(cv_r2_scores) if cv_r2_scores else 0
        
        # AIC and BIC
        n = len(y_data)
        k = len(params)
        rss = np.sum(residuals**2)
        aic = n * np.log(rss/n) + 2*k
        bic = n * np.log(rss/n) + k*np.log(n)
        
        # Store all results
        results[model_name] = {
            'params': params,
            'cov': cov,
            'predictions': y_pred,
            'residuals': residuals,
            'residuals_norm': residuals_norm,
            'r2': r2,
            'rmse': rmse,
            'mae': mae,
            'cv_r2_mean': cv_r2_mean,
            'cv_r2_std': cv_r2_std,
            'aic': aic,
            'bic': bic,
            # Distribution tests
            'ks_stat': ks_stat,
            'ks_pvalue': ks_pvalue,
            'ad_stat': ad_result.statistic,
            'ad_critical': ad_result.critical_values[2],  # 5% level
            'ad_pass': ad_pass,
            'chi2_stat': chi2_stat,
            'chi2_pvalue': chi2_pvalue,
            'sw_stat': sw_stat,
            'sw_pvalue': sw_pvalue,
            'dw_stat': dw_stat,
            'runs_stat': runs_stat,
            'runs_pvalue': runs_pvalue,
            'residual_std': residual_std
        }
        
        print(f"\n✓ Successfully fitted {model_name} model")
        
    except Exception as e:
        print(f"\n✗ Failed to fit {model_name}: {str(e)}")
        results[model_name] = None

# ============================================================================
# STEP 3: Comprehensive Model Comparison
# ============================================================================
print(f"\n{'='*80}")
print("COMPREHENSIVE MODEL COMPARISON")
print(f"{'='*80}\n")

# Create detailed comparison table
comparison_data = []
for name, result in results.items():
    if result is not None:
        comparison_data.append({
            'Model': name,
            'R²': result['r2'],
            'RMSE': result['rmse'],
            'AIC': result['aic'],
            'BIC': result['bic'],
            'K-S p': result['ks_pvalue'],
            'A-D': '✓' if result['ad_pass'] else '✗',
            'χ² p': result['chi2_pvalue'],
            'S-W p': result['sw_pvalue'],
            'D-W': result['dw_stat'],
            'Pass': sum([
                result['ks_pvalue'] > 0.05,
                result['ad_pass'],
                result['chi2_pvalue'] > 0.05 if not np.isnan(result['chi2_pvalue']) else 0,
                result['sw_pvalue'] > 0.05,
                1.5 < result['dw_stat'] < 2.5
            ])
        })

# Check if we have any successful fits
if len(comparison_data) == 0:
    print("⚠ WARNING: No models successfully fitted!")
    print("This may be due to:")
    print("  - Insufficient or invalid data")
    print("  - Inappropriate model specifications")
    print("  - Numerical instabilities during fitting")
else:
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('Pass', ascending=False)
    
    print("\n" + "="*100)
    print("STATISTICAL TEST SUMMARY")
    print("="*100)
    print(comparison_df.to_string(index=False))
    print("\nLegend:")
    print("  K-S p   : Kolmogorov-Smirnov p-value (>0.05 = pass)")
    print("  A-D     : Anderson-Darling test (✓ = pass)")
    print("  χ² p    : Chi-square p-value (>0.05 = pass)")
    print("  S-W p   : Shapiro-Wilk p-value (>0.05 = pass)")
    print("  D-W     : Durbin-Watson statistic (1.5-2.5 = no autocorrelation)")
    print("  Pass    : Number of tests passed (max = 5)")
    
    # Select best model
    best_model_idx = comparison_df['Pass'].idxmax()
    best_model_name = comparison_df.loc[best_model_idx, 'Model']
    best_r2 = comparison_df.loc[best_model_idx, 'R²']
    
    print(f"\n{'='*80}")
    print(f"🏆 BEST MODEL: {best_model_name}")
    print(f"{'='*80}")
    print(f"   R² Score:          {best_r2:.6f}")
    print(f"   Tests Passed:      {comparison_df.loc[best_model_idx, 'Pass']}/5")
    print(f"   AIC:               {comparison_df.loc[best_model_idx, 'AIC']:.2f}")
    print(f"   K-S p-value:       {comparison_df.loc[best_model_idx, 'K-S p']:.6e}")
    print(f"   Anderson-Darling:  {comparison_df.loc[best_model_idx, 'A-D']}")
    print(f"   Chi-square p:      {comparison_df.loc[best_model_idx, 'χ² p']:.6e}")
    
    # Save comparison results
    file_path_results = os.path.join(PATH, data_folder, results_folder, cs_folder, data_folder)
    comparison_save_path = os.path.join(file_path_results, "model_comparison_comprehensive.csv")
    comparison_df.to_csv(comparison_save_path, index=False)
    print(f"\n✓ Saved comprehensive comparison to: {comparison_save_path}")

print(f"\n{'='*80}")
print("COMPREHENSIVE VALIDATION COMPLETE")
print(f"{'='*80}\n")